# **Anime Hybrid Recommendation System**

## **Dataset Description**

## **Anime Dataset 2023 Dataset**

### **Summary**:

Anime is a popular form of Japanese animated entertainment known for its unique art style, diverse genres, and rich storytelling. It includes TV series, movies, and OVAs that appeal to a wide range of audiences worldwide. With thousands of titles spanning action, romance, fantasy, and more, anime has built a passionate global fanbase. This diversity makes anime data especially suitable for building a great recommendation system. This dataset(**[Anime Dataset 2023](https://www.kaggle.com/datasets/dbdmobile/myanimelist-dataset?select=anime-dataset-2023.csv)**) contains detailed metadata for each anime entry, including identifiers, content details, ratings, and user interaction metrics. The following are the descriptions of each column:

---

### **Column Descriptions**:

* **`anime_id`**: Unique identifier for each anime.
* **`Name`**: Original name of the anime.
* **`English name`**: Official English-translated title.
* **`Other name`**: Alternate titles in native languages (e.g., Japanese, Chinese, Korean).
* **`Score`**: Average user rating for the anime.
* **`Genres`**: Comma-separated list of genres associated with the anime.
* **`Synopsis`**: Brief summary of the anime’s plot.
* **`Type`**: Format of the anime (e.g., TV, Movie, OVA).
* **`Episodes`**: Total number of episodes.
* **`Aired`**: Airing date range (start to end).
* **`Premiered`**: Season and year of initial release.
* **`Status`**: Current airing status (e.g., Finished Airing, Currently Airing).
* **`Producers`**: Companies involved in the production.
* **`Licensors`**: Distribution or licensing companies (e.g., streaming platforms).
* **`Studios`**: Animation studios that created the anime.
* **`Source`**: Origin of the story (e.g., manga, novel, original).
* **`Duration`**: Length of a single episode.
* **`Rating`**: Age restriction or content rating (e.g., PG-13, R).
* **`Rank`**: Position in ranking based on ratings or popularity.
* **`Popularity`**: Popularity rank among all anime.
* **`Favorites`**: Number of users who marked the anime as a favorite.
* **`Scored By`**: Number of users who rated the anime.
* **`Members`**: Total users who added the anime to their list (watching, completed, etc.).
* **`Image URL`**: Link to the anime’s cover image or poster.

## **Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Installations**

In [ ]:
!pip install numpy==1.26.4 --force-reinstall --no-cache-dir


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 187.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
!pip install surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl size=2469541 sha256=4435555033e13403a54af1dec61f311298d670776920c86a305030189ace65b9
  Stored in directory: /root/.cache/pip/wheels/2a/8f/6e/7e2899163e2d85d8266daab4aa1cdabec7a6c56f83c015b5af
Successfully built scikit-surprise


## **Imports**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import Image, HTML
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import string
import spacy
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from surprise import SVD

## **Data Understanding**

### **Data Loading**

* **Use pathlib for `path` safety**

In [ ]:
data_path = Path('/content/drive/MyDrive/Anime Recommender System')

* **Loading Anime Data**

In [ ]:
anime_df = pd.read_csv(data_path/'anime-dataset-2023.csv')

In [ ]:
anime_df.columns

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='object')

* **Showing first 5 rows**

In [ ]:
anime_df.head()

,anime_id,Name,English name,Other name,Score,Genres,Synopsis,Type,Episodes,Aired,...,Studios,Source,Duration,Rating,Rank,Popularity,Favorites,Scored By,Members,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,カウボーイビバップ,8.75,"Action, Award Winning, Sci-Fi","Crime is timeless. By the year 2071, humanity ...",TV,26.0,"Apr 3, 1998 to Apr 24, 1999",...,Sunrise,Original,24 min per ep,R - 17+ (violence & profanity),41.0,43,78525,914193.0,1771505,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,カウボーイビバップ 天国の扉,8.38,"Action, Sci-Fi","Another day, another bounty—such is the life o...",Movie,1.0,"Sep 1, 2001",...,Bones,Original,1 hr 55 min,R - 17+ (violence & profanity),189.0,602,1448,206248.0,360978,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,トライガン,8.22,"Action, Adventure, Sci-Fi","Vash the Stampede is the man with a $$60,000,0...",TV,26.0,"Apr 1, 1998 to Sep 30, 1998",...,Madhouse,Manga,24 min per ep,PG-13 - Teens 13 or older,328.0,246,15035,356739.0,727252,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),7.25,"Action, Drama, Mystery, Supernatural",Robin Sena is a powerful craft user drafted in...,TV,26.0,"Jul 3, 2002 to Dec 25, 2002",...,Sunrise,Original,25 min per ep,PG-13 - Teens 13 or older,2764.0,1795,613,42829.0,111931,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,冒険王ビィト,6.94,"Adventure, Fantasy, Supernatural",It is the dark century and the people are suff...,TV,52.0,"Sep 30, 2004 to Sep 29, 2005",...,Toei Animation,Manga,23 min per ep,PG - Children,4240.0,5126,14,6413.0,15001,https://cdn.myanimelist.net/images/anime/7/215...


* **Loading Ratings Data**

In [ ]:
ratings_df = pd.read_csv(data_path/'users-score-2023.csv')[:1000]

* **Showing first 5 rows of the data**

In [ ]:
ratings_df.head()

,user_id,Username,anime_id,Anime Title,rating
0,1,Xinil,21,One Piece,9
1,1,Xinil,48,.hack//Sign,7
2,1,Xinil,320,A Kite,5
3,1,Xinil,49,Aa! Megami-sama!,8
4,1,Xinil,304,Aa! Megami-sama! Movie,8


## **Data Exploration**

### **Anime data size**

In [ ]:
anime_df.shape

(24905, 24)

 * The anime dataset contains `24,905` entries (rows) and `24` features (columns)

### **User Ratings data size**

In [ ]:
ratings_df.shape

(24325191, 5)

* The ratings dataset contains `24,325,191` records and `5` columns.
* This large volume of `user-anime` interaction data provides a strong foundation for `Collaborative Filtering` and other recommendation techniques. The richness and scale of the dataset make it ideal for training robust models.

### **Check for Missing Values in Anime Dataset**

In [ ]:
anime_df.isnull().sum()

,0
anime_id,0
Name,0
English name,0
Other name,0
Score,0
Genres,0
Synopsis,0
Type,0
Episodes,0
Aired,0


* No missing values.

### **Remove commas or other non-numeric characters from `Score` column (if any)**

Cleaning the `'Score'` and `'Scored By'` columns in the anime dataset by removing non-numeric characters using regular expressions:

* **`'Score'`**: Removes any character that is not a digit or a decimal point (e.g., "N/A", text, etc.).
* **`'Scored By'`**: Removes all characters except digits to ensure the field contains only numeric values.

In [ ]:
anime_df['Score'] = anime_df['Score'].replace('[^0-9.]', '', regex=True)
anime_df['Scored By'] = anime_df['Scored By'].replace('[^0-9]', '', regex=True)

### **Convert to Numeric**

In [ ]:
anime_df['Score'] = pd.to_numeric(anime_df['Score'], errors='coerce')
anime_df['Scored By'] = pd.to_numeric(anime_df['Scored By'], errors='coerce')

### **Fill missing values with the median of each column**

In [ ]:
anime_df['Score'].fillna(anime_df['Score'].median(), inplace=True)
anime_df['Scored By'].fillna(anime_df['Scored By'].median(), inplace=True)

### **Extract the first 4-digit number from `Aired` as the release year**

In [ ]:
anime_df['release_year'] = anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

### **Split genres, handle `Unknown`**

In [ ]:
anime_df['Genres'] = anime_df['Genres'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])

### **Multi-hot encode**

In [ ]:
mlb_genres = MultiLabelBinarizer()
genres_encoded = mlb_genres.fit_transform(anime_df['Genres'])

### **Split studios, handle `Unknown`**

In [ ]:
anime_df['Studios'] = anime_df['Studios'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])

### **Multi-hot encode**

In [ ]:
mlb_studios = MultiLabelBinarizer()
studios_encoded = mlb_studios.fit_transform(anime_df['Studios'])

### **Add weighted Rating feature which is the IMDB popularity function quotient**

In [ ]:
C = anime_df['Score'].mean()
m = anime_df['Scored By'].quantile(0.65)

anime_df['weighted_rating'] = (
    (anime_df['Scored By'] / (anime_df['Scored By'] + m)) * anime_df['Score'] +
    (m / (anime_df['Scored By'] + m)) * C
)


### **One-hot encode `Type` feature**

In [ ]:
ohe_type = OneHotEncoder(sparse_output=False)
type_encoded = ohe_type.fit_transform(anime_df[['Type']])

### **One-hot encode `Source` feature**

In [ ]:
ohe_source = OneHotEncoder(sparse_output=False)
source_encoded = ohe_source.fit_transform(anime_df[['Source']])

### **Convert to numeric, coercing errors to `NaN`**

In [ ]:
anime_df['Episodes'] = pd.to_numeric(anime_df['Episodes'], errors='coerce')

### **Impute missing values with the median**

In [ ]:
median_episodes = anime_df['Episodes'].median()
anime_df['Episodes'].fillna(median_episodes, inplace=True)

### **Bin episodes**

In [ ]:
bins = [0, 1, 12, 24, 50, np.inf]
labels = ['1', '2-12', '13-24', '25-50', '51+']
anime_df['Episodes_Binned'] = pd.cut(anime_df['Episodes'], bins=bins, labels=labels)

### **One-hot encode `Episodes` feature**

In [ ]:
ohe_episodes = OneHotEncoder(sparse_output=False)
episodes_encoded = ohe_episodes.fit_transform(anime_df[['Episodes_Binned']])

### **Clean text function**

In [ ]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
  return text

#### **Apply cleaning**

In [ ]:
anime_df['Synopsis'] = anime_df['Synopsis'].apply(clean_text)

#### **TF-IDF vectorization**

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
synopsis_encoded = tfidf.fit_transform(anime_df['Synopsis'])

#### **Feature Selection**

In [ ]:
anime_df = anime_df[ [
    'anime_id',
    'Name',
    'Score',
    'Genres',
    'Synopsis',
    'Type',
    'Episodes',
    'Aired',
    'Status',
    'Studios',
    'Source',
    'Scored By',
    'Image URL'
]]

* **Save the new df**

In [ ]:
# Define the path and filename
output_path = '/content/drive/MyDrive/Anime Recommender System/anime_filtered.csv'

# Save the DataFrame to CSV
anime_df.to_csv(output_path, index=False)

## **Building Content-Based Recommendation System**

#### **Compute cosine similarities with sparse matrices**

Computing **cosine similarity matrices** for different encoded content features of anime:

* **`genres_sim`**: Measures how similar anime titles are based on genre vectors.
* **`studios_sim`**: Measures similarity based on shared animation studios.
* **`synopsis_sim`**: Measures textual similarity between anime plot summaries (e.g., using TF-IDF).

Cosine similarity returns values between `0` (completely dissimilar) and `1` (identical), making it ideal for comparing sparse or high-dimensional feature encodings.

In [ ]:
genres_sim = cosine_similarity(genres_encoded)
studios_sim = cosine_similarity(studios_encoded)
synopsis_sim = cosine_similarity(synopsis_encoded)

### **Binary Similarity Match Function**


This function computes a **binary similarity matrix** for an encoded dataset using **dot product matching**:

* It calculates whether two entries share at least one common feature.
* If they do, the dot product is non-zero → `True` → converted to `1`.
* If no match is found → `False` → converted to `0`.

The result is a symmetric **binary matrix** (`1` = match, `0` = no match), useful for exact-match filtering (e.g., genre or studio overlap).

In [ ]:
def match_similarity(encoded_data):
    return (encoded_data @ encoded_data.T).astype(bool).astype(int)

### **Compute Exact Match Similarities (Type, Source, Episodes)**

The `match_similarity()` function to compute **binary similarity matrices** for the following encoded features:

* **`type_sim`**: Matches anime with the same format (e.g., TV, Movie, OVA).
* **`source_sim`**: Matches anime with the same source material (e.g., Manga, Light Novel, Original).
* **`episodes_sim`**: Matches anime with the same number (or encoded range) of episodes.

Each matrix contains:

* `1` → at least one shared encoded value (i.e., a match)
* `0` → no match

These matrices are useful for **filtering or boosting** recommendations that share structural traits with the input anime.


In [ ]:
type_sim = match_similarity(type_encoded)
source_sim = match_similarity(source_encoded)
episodes_sim = match_similarity(episodes_encoded)

### **Combine Feature-Based Similarities with Equal Weights**

Combining multiple similarity matrices into a single **composite similarity score** by applying **equal weights** to each of the six selected features:

* **Features used**:

  1. `genres_sim`
  2. `synopsis_sim`
  3. `type_sim`
  4. `studios_sim`
  5. `episodes_sim`
  6. `source_sim`

* Each feature is assigned an equal weight of `1/6`, assuming equal importance across all content aspects.

The resulting **`combined_sim` matrix** represents a **hybrid similarity score** that captures both **semantic** (synopsis, genres) and **structural** (type, source, etc.) similarities — useful for **content-based recommendations**.

In [ ]:
# Number of features
num_features = 6

# Equal weights (1/6 for each)
weights = [1 / num_features] * num_features

# Combine similarities
combined_sim = (weights[0] * genres_sim +
                weights[1] * synopsis_sim +
                weights[2] * type_sim +
                weights[3] * studios_sim +
                weights[4] * episodes_sim +
                weights[5] * source_sim)

### **Hybrid Anime Recommendation Function (Content-Based + Weighted Rating)**

The followig function, `get_recommendations2()`, generates personalized anime recommendations based on **content similarity** and **weighted rating scores**.

#### **Parameters**:

* `title` (*str*): Anime title to base the recommendations on.
* `n` (*int*): Number of recommendations to return (default = 10).
* `similarity_weight` (*float*): Controls how much weight to give similarity vs. rating in final scoring.

#### **How it works**:

1. **Locates** the given anime in `anime_df`.
2. **Computes cosine similarity scores** using the `combined_sim` matrix.
3. **Selects top N most similar titles**, excluding the anime itself.
4. **Fetches relevant columns**: name, genres, image, scores, etc.
5. **Merges similarity scores** with existing weighted ratings.
6. **Calculates a final score** using a weighted average:
$$\text{final_score} = (1 - w)\cdot\text{weighted_rating} + w \cdot \text{similarity_score}$$
7. **Returns top N** results sorted by this final score.


In [ ]:
def get_recommendations2(title, n=10, similarity_weight=0.85):
    """
    Recommend anime based on a given title using cosine similarity and weighted ratings.

    Parameters:
    - title (str): Name of the anime to base recommendations on.
    - n (int): Number of recommendations to return (default: 10).
    - similarity_weight (float): Weight for similarity in final score (default: 0.7).

    Returns:
    - DataFrame: Top N recommended animes with relevant details.

    Raises:
    - ValueError: If the title is not found in anime_df.
    """
    # Check if title exists and get its label index
    matching_animes = anime_df[anime_df['Name'] == title]
    if matching_animes.empty:
        raise ValueError(f"Anime '{title}' not found in the database.")
    label = matching_animes.index[0]

    # Convert label index to positional index
    pos = anime_df.index.get_loc(label)

    # Get pairwise similarity scores
    sim_scores = list(enumerate(combined_sim[pos]))

    # Sort by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top-N matches (excluding self), adjust n if fewer animes exist
    max_recommendations = min(102, len(anime_df) - 1)
    top_sim = sim_scores[1:max_recommendations + 1]
    sim_indices = [i for i, _ in top_sim]  # Indices of top similar animes
    sim_scores_top = [score for _, score in top_sim]  # Corresponding similarity scores

    # Fetch recommended animes
    recommended_animes = anime_df.iloc[sim_indices][
        ['Name', 'anime_id', 'weighted_rating', 'Image URL', 'Type', 'Genres', 'Score']
    ].copy()

    # Prepare for merging
    qualified_animes = recommended_animes.copy()
    qualified_animes.reset_index(drop=True, inplace=True)

    # Create similarity DataFrame with correct scores
    similarity_df = pd.DataFrame({
        "anime_id": anime_df.iloc[sim_indices]["anime_id"].values,
        "similarity_score": sim_scores_top
    })

    # Merge on correct column
    qualified_animes = qualified_animes.merge(similarity_df, on="anime_id", how="left")

    # Handle any unexpected NaNs in similarity_score
    qualified_animes["similarity_score"] = qualified_animes["similarity_score"].fillna(
        qualified_animes["similarity_score"].min()
    )

    # Calculate final score
    qualified_animes['final_score'] = (
        (1 - similarity_weight) * qualified_animes['weighted_rating'] +
        similarity_weight * qualified_animes['similarity_score']
    )

    # Return top-N sorted by final score
    return qualified_animes.sort_values('final_score', ascending=False).head(n)[
        ['Name', 'anime_id', 'Image URL', 'similarity_score', 'weighted_rating', 'final_score']
    ]

* **Lets test it.**

In [ ]:
get_recommendations2(title='Darling in the FranXX', n=24)

,Name,anime_id,Image URL,similarity_score,weighted_rating,final_score
1,Lycoris Recoil,50709,https://cdn.myanimelist.net/images/anime/1392/...,0.684209,8.183950,1.809170
67,Vivy: Fluorite Eye's Song,46095,https://cdn.myanimelist.net/images/anime/1637/...,0.597510,8.394183,1.767011
8,Kill la Kill,18679,https://cdn.myanimelist.net/images/anime/1464/...,0.647670,8.036835,1.756045
77,Psycho-Pass,13601,https://cdn.myanimelist.net/images/anime/5/433...,0.587892,8.335466,1.750028
14,Plastic Memories,27775,https://cdn.myanimelist.net/images/anime/4/727...,0.645286,7.904120,1.734111
30,Senki Zesshou Symphogear XV,32843,https://cdn.myanimelist.net/images/anime/1899/...,0.620386,7.975510,1.723654
12,Texhnolyze,26,https://cdn.myanimelist.net/images/anime/1027/...,0.646544,7.717234,1.707147
33,Carole & Tuesday,37435,https://cdn.myanimelist.net/images/anime/1611/...,0.620216,7.851340,1.704885
53,Mahou Shoujo Lyrical Nanoha A's,77,https://cdn.myanimelist.net/images/anime/4/676...,0.599961,7.889765,1.693432
13,Uchuu Patrol Luluco,32681,https://cdn.myanimelist.net/images/anime/4/790...,0.645950,7.512696,1.675962


In [ ]:
df = get_recommendations2(title='Darling in the FranXX', n=67)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Name,anime_id,Image URL,similarity_score,weighted_rating,final_score
1,Lycoris Recoil,50709,,0.684209,8.183950,1.809170
67,Vivy: Fluorite Eye's Song,46095,,0.597510,8.394183,1.767011
8,Kill la Kill,18679,,0.647670,8.036835,1.756045
77,Psycho-Pass,13601,,0.587892,8.335466,1.750028
14,Plastic Memories,27775,,0.645286,7.904120,1.734111
30,Senki Zesshou Symphogear XV,32843,,0.620386,7.975510,1.723654
12,Texhnolyze,26,,0.646544,7.717234,1.707147
33,Carole & Tuesday,37435,,0.620216,7.851340,1.704885
53,Mahou Shoujo Lyrical Nanoha A's,77,,0.599961,7.889765,1.693432
13,Uchuu Patrol Luluco,32681,,0.645950,7.512696,1.675962


In [ ]:
get_recommendations2(title='Detective Conan', n=24)

,Name,anime_id,Image URL,similarity_score,weighted_rating,final_score
2,Dr. Stone: New World,48549,https://cdn.myanimelist.net/images/anime/1316/...,0.777778,8.205729,1.891970
1,Dr. Stone: Stone Wars,40852,https://cdn.myanimelist.net/images/anime/1711/...,0.778460,8.164181,1.886318
10,Kamisama Hajimemashita◎,25681,https://cdn.myanimelist.net/images/anime/8/691...,0.724089,8.194550,1.844658
53,Mushishi Zoku Shou 2nd Season,24701,https://cdn.myanimelist.net/images/anime/9/680...,0.599115,8.689948,1.812740
70,Mushishi Zoku Shou,21939,https://cdn.myanimelist.net/images/anime/13/58...,0.598093,8.664767,1.808094
49,Kaguya-sama wa Kokurasetai? Tensai-tachi no Re...,40591,https://cdn.myanimelist.net/images/anime/1764/...,0.599410,8.635077,1.804760
58,Grand Blue,37105,https://cdn.myanimelist.net/images/anime/1302/...,0.598806,8.420849,1.772112
39,Kaguya-sama wa Kokurasetai: Tensai-tachi no Re...,37999,https://cdn.myanimelist.net/images/anime/1295/...,0.600087,8.406546,1.771056
64,Karakai Jouzu no Takagi-san 3,49721,https://cdn.myanimelist.net/images/anime/1861/...,0.598334,8.382613,1.765976
94,Tensei shitara Slime Datta Ken 2nd Season,39551,https://cdn.myanimelist.net/images/anime/1271/...,0.596752,8.382957,1.764683


In [ ]:
df = get_recommendations2(title='Hunter x Hunter (2011)', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Name,anime_id,Image URL,similarity_score,weighted_rating,final_score
22,Fullmetal Alchemist: Brotherhood,5114,,0.647648,9.097636,1.915146
17,Monster,19,,0.666667,8.858186,1.895395
2,Hunter x Hunter,136,,0.731087,8.397139,1.880995
9,Hajime no Ippo,263,,0.667818,8.744465,1.879315
1,Cardcaptor Sakura,232,,0.756345,8.145421,1.864706
35,Yuu☆Yuu☆Hakusho,392,,0.641396,8.448490,1.812460
6,Naruto: Shippuuden,1735,,0.669019,8.257899,1.807351
5,Diamond no Ace: Act II,38731,,0.669196,8.184648,1.796514
28,Dragon Ball Z,813,,0.644659,8.156166,1.771385
14,D.Gray-man,1482,,0.666992,8.009810,1.768414


#### **Conclusion**
The recommendation results for Darling in the FranXX and Detective Conan show that the system effectively identifies thematically similar anime by combining content similarity and weighted user ratings. The suggestions align well with genre, tone, and narrative style, demonstrating that the hybrid model captures both semantic and structural relevance. However, most of the top results tend to favor well-established and popular titles.

To improve diversity and promote discovery, the system can be enhanced by giving priority to newer anime titles, such as by adjusting the weighting formula to favor recent release years or applying a novelty/recency boost to the final score.

### **Lets do some improvements**

### **Generate Recency Score Based on Release Year**

This helper function adds a **`recency_score`** column to a DataFrame using the anime's release year:

* It creates a numeric score between **0 and 1**, where:

  * `0` represents the **oldest anime**
  * `1` represents the **most recent anime**
* Missing values in `release_year` are replaced with the **minimum year** to avoid distortion.
* This normalized score can be used to **boost newer anime** in recommendation results, helping balance popularity with recency and discovery.

This is a key step in **promoting fresh or underrated titles** within the system.

In [ ]:
def _apply_recency_boost(df):
    """
    Uses the numeric 'release_year' column to create a 0–1 scaled 'recency_score'.
    """
    # Use 'release_year' directly
    df['year'] = df['release_year']

    # Fallback for missing years (fill with oldest year)
    df['year'] = df['year'].fillna(df['year'].min())

    # Scale to [0, 1]
    min_year = df['year'].min()
    max_year = df['year'].max()
    year_range = max_year - min_year if max_year != min_year else 1

    df['recency_score'] = (df['year'] - min_year) / year_range

    return df


### **Hybrid Recommendation Function with Recency Boost**

This enhanced version of the recommendation function combines:

1. **Content similarity** (from hybrid features like genres, synopsis, etc.)
2. **Weighted user ratings**
3. **Recency score** (based on the anime's release year)

#### **How It Works**:

* Computes similarity scores using `combined_sim`
* Applies `_apply_recency_boost()` to normalize the anime's release year to a `recency_score` (0–1)
* Combines:

  * `similarity_score` (e.g., 85%)
  * `weighted_rating` (remaining weight)
  * `recency_score` (separate weight layered on top)

This setup promotes **newer titles** while still prioritizing relevance and quality.

#### **Returns**:

A DataFrame with:

* Anime name, ID, image
* Similarity score, weighted rating, recency score, final score, and release year

In [ ]:
def get_recommendations3(title, n=10, similarity_weight=0.85, recency_weight=0.3):
    """
    Recommend anime based on a given title using cosine similarity, weighted ratings, and recency.

    Parameters:
    - title (str): Name of the anime to base recommendations on.
    - n (int): Number of recommendations to return (default: 10).
    - similarity_weight (float): Weight for similarity in final score (default: 0.85).
    - recency_weight (float): Weight for recency in final score (default: 0.1).

    Returns:
    - DataFrame: Top N recommended animes with relevant details.

    Raises:
    - ValueError: If the title is not found in anime_df or weights are invalid.
    """
    # Check if title exists and get its label index
    matching_animes = anime_df[anime_df['Name'] == title]
    if matching_animes.empty:
        raise ValueError(f"Anime '{title}' not found in the database.")
    label = matching_animes.index[0]

    # Convert label index to positional index
    pos = anime_df.index.get_loc(label)

    # Get pairwise similarity scores
    sim_scores = list(enumerate(combined_sim[pos]))

    # Sort by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top-N matches (excluding self), adjust n if fewer animes exist
    max_recommendations = min(102, len(anime_df) - 1)
    top_sim = sim_scores[1:max_recommendations + 1]
    sim_indices = [i for i, _ in top_sim]  # Indices of top similar animes
    sim_scores_top = [score for _, score in top_sim]  # Corresponding similarity scores

    # Fetch recommended animes, including Release_date for recency
    recommended_animes = anime_df.iloc[sim_indices][
        ['Name', 'anime_id', 'weighted_rating', 'Image URL', 'Type', 'Genres', 'Score', 'release_year']
    ].copy()

    # Apply recency boost to add recency_score
    recommended_animes = _apply_recency_boost(recommended_animes)

    # Prepare for merging
    qualified_animes = recommended_animes.copy()
    qualified_animes.reset_index(drop=True, inplace=True)

    # Create similarity DataFrame with correct scores
    similarity_df = pd.DataFrame({
        "anime_id": anime_df.iloc[sim_indices]["anime_id"].values,
        "similarity_score": sim_scores_top
    })

    # Merge on correct column
    qualified_animes = qualified_animes.merge(similarity_df, on="anime_id", how="left")

    # Handle any unexpected NaNs in similarity_score
    qualified_animes["similarity_score"] = qualified_animes["similarity_score"].fillna(
        qualified_animes["similarity_score"].min()
    )

    # Calculate the weight for weighted_rating
    rating_weight = 1 - similarity_weight
    if rating_weight < 0:
        raise ValueError("The sum of similarity_weight and recency_weight must be less than or equal to 1.")

    # Calculate final score with recency
    qualified_animes['final_score'] = (
        rating_weight * qualified_animes['weighted_rating'] +
        similarity_weight * qualified_animes['similarity_score'])

    qualified_animes['recency_score'] = recency_weight * qualified_animes['recency_score'] + (1 - recency_weight) * qualified_animes['final_score']

    # Return top-N sorted by final score
    return qualified_animes.sort_values('final_score', ascending=False).head(n)[
        ['Name', 'anime_id', 'Image URL', 'similarity_score', 'weighted_rating', 'recency_score','release_year', 'final_score']
    ]

#### Lets show some tests.

In [ ]:
df = get_recommendations3(title='Hunter x Hunter (2011)', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Name,anime_id,Image URL,similarity_score,weighted_rating,recency_score,release_year,final_score
22,Fullmetal Alchemist: Brotherhood,5114,,0.647648,9.097636,1.570602,2009.0,1.915146
17,Monster,19,,0.666667,8.858186,1.531776,2004.0,1.895395
2,Hunter x Hunter,136,,0.731087,8.397139,1.496697,1999.0,1.880995
9,Hajime no Ippo,263,,0.667818,8.744465,1.500521,2000.0,1.879315
1,Cardcaptor Sakura,232,,0.756345,8.145421,1.480294,1998.0,1.864706
35,Yuu☆Yuu☆Hakusho,392,,0.641396,8.448490,1.413722,1992.0,1.812460
6,Naruto: Shippuuden,1735,,0.669019,8.257899,1.485146,2007.0,1.807351
5,Diamond no Ace: Act II,38731,,0.669196,8.184648,1.537560,2019.0,1.796514
28,Dragon Ball Z,813,,0.644659,8.156166,1.369969,1989.0,1.771385
14,D.Gray-man,1482,,0.666992,8.009810,1.452890,2006.0,1.768414


In [ ]:
get_recommendations3(title='Hunter x Hunter (2011)', n=24)

,Name,anime_id,Image URL,similarity_score,weighted_rating,recency_score,release_year,final_score
22,Fullmetal Alchemist: Brotherhood,5114,https://cdn.myanimelist.net/images/anime/1208/...,0.647648,9.097636,1.570602,2009.0,1.915146
17,Monster,19,https://cdn.myanimelist.net/images/anime/10/18...,0.666667,8.858186,1.531776,2004.0,1.895395
2,Hunter x Hunter,136,https://cdn.myanimelist.net/images/anime/1305/...,0.731087,8.397139,1.496697,1999.0,1.880995
9,Hajime no Ippo,263,https://cdn.myanimelist.net/images/anime/4/863...,0.667818,8.744465,1.500521,2000.0,1.879315
1,Cardcaptor Sakura,232,https://cdn.myanimelist.net/images/anime/8/607...,0.756345,8.145421,1.480294,1998.0,1.864706
35,Yuu☆Yuu☆Hakusho,392,https://cdn.myanimelist.net/images/anime/1228/...,0.641396,8.448490,1.413722,1992.0,1.812460
6,Naruto: Shippuuden,1735,https://cdn.myanimelist.net/images/anime/1565/...,0.669019,8.257899,1.485146,2007.0,1.807351
5,Diamond no Ace: Act II,38731,https://cdn.myanimelist.net/images/anime/1153/...,0.669196,8.184648,1.537560,2019.0,1.796514
28,Dragon Ball Z,813,https://cdn.myanimelist.net/images/anime/1607/...,0.644659,8.156166,1.369969,1989.0,1.771385
14,D.Gray-man,1482,https://cdn.myanimelist.net/images/anime/13/75...,0.666992,8.009810,1.452890,2006.0,1.768414


#### **Conclusion**:
Adding a recency score to the recommendation system introduces a valuable balance between relevance and novelty. While traditional hybrid models tend to favor older, highly rated classics, integrating recency allows newer anime to surface more prominently without sacrificing quality. This improves content discovery, keeps recommendations timely, and better reflects user interest in modern trends.

To further enhance the quality of recommendations, it is essential to fine-tune the weights assigned to each content feature (e.g., genres, synopsis, studio, type). While equal weighting provides a neutral baseline, not all features contribute equally to user preferences. For instance, genres and synopsis often carry more semantic relevance than production studios or episode count. By adjusting these weights based on empirical performance or user feedback, the system can better align with real-world viewing behavior, resulting in more accurate, engaging, and personalized recommendations.

### **lets adjust the weights of the features**


In [ ]:
# New weights based on user preference
weights = [0.35, 0.25, 0.15, 0.10, 0.075, 0.075, 0.05]

# Combine similarities
combined_sim2 = (weights[0] * synopsis_sim +
                weights[1] * genres_sim +
                weights[2] * type_sim +
                weights[3] * studios_sim +
                weights[4] * episodes_sim +
                weights[5] * source_sim)

This function refines anime recommendations by introducing a **flexible weighting scheme** for similarity, rating, and recency. It uses a new similarity matrix (`combined_sim2`) generated from **custom-weighted feature similarities**, allowing more control over what influences the recommendations.

#### **How It Works**:

1. Retrieves anime similar to the given `title` using the **custom similarity matrix**.
2. Calculates a **`recency_score`** scaled from 0–1 based on release year.
3. Computes a final score based on:

   * `similarity_weight` (semantic closeness)
   * `weighted_rating` (user consensus)
   * `recency_weight` (favoring newer content)
4. Applies a **composite scoring formula**:

   $$\text{final_score} = \left[ \text{rating} \cdot w_r + \text{similarity} \cdot w_s \right] \cdot (1 - w_{rec}) + \text{recency} \cdot w_{rec}$$

   where $w_r + w_s + w_{rec} = 1$

#### **Explaination**:

Feature importance isn't uniform—**genres and synopsis** usually reflect content better than **studio or type**. By customizing the weights used in the similarity matrix (`combined_sim2`), the system can prioritize features that **align more closely with user preferences**, improving both **accuracy** and **relevance** of recommendations.

In [ ]:
def get_recommendations4(title, n=10, similarity_weight=0.7, recency_weight=0.1):
    # Check if title exists and get its label index
    matching_animes = anime_df[anime_df['Name'] == title]
    if matching_animes.empty:
        raise ValueError(f"Anime '{title}' not found in the database.")
    label = matching_animes.index[0]

    pos = anime_df.index.get_loc(label)
    sim_scores = list(enumerate(combined_sim2[pos]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    max_recommendations = min(102, len(anime_df) - 1)
    top_sim = sim_scores[1:max_recommendations + 1]
    sim_indices = [i for i, _ in top_sim]
    sim_scores_top = [score for _, score in top_sim]

    recommended_animes = anime_df.iloc[sim_indices][
        ['Name', 'anime_id', 'weighted_rating', 'Image URL', 'Type', 'Genres', 'Score', 'release_year']
    ].copy()

    recommended_animes = _apply_recency_boost(recommended_animes)
    qualified_animes = recommended_animes.copy()
    qualified_animes.reset_index(drop=True, inplace=True)

    similarity_df = pd.DataFrame({
        "anime_id": anime_df.iloc[sim_indices]["anime_id"].values,
        "similarity_score": sim_scores_top
    })

    qualified_animes = qualified_animes.merge(similarity_df, on="anime_id", how="left")
    qualified_animes["similarity_score"] = qualified_animes["similarity_score"].fillna(
        qualified_animes["similarity_score"].min()
    )

    rating_weight = 1 - similarity_weight - recency_weight
    if rating_weight < 0:
        raise ValueError("The sum of similarity_weight and recency_weight must be less than or equal to 1.")

    qualified_animes['final_score'] = (
        rating_weight * qualified_animes['weighted_rating'] +
        similarity_weight * qualified_animes['similarity_score'])*(1-recency_weight) +(
        recency_weight * qualified_animes['recency_score']
    )

    return qualified_animes.sort_values('final_score', ascending=False).head(n)[
        ['Image URL', 'Name', 'anime_id',  'similarity_score', 'weighted_rating', 'recency_score', 'release_year', 'final_score']
    ]

* **lets do somme tests**

In [ ]:
df = get_recommendations4(title='Hunter x Hunter', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
0,,Hunter x Hunter (2011),11061,0.685284,9.037173,0.793103,2011.0,2.137730
23,,Fullmetal Alchemist: Brotherhood,5114,0.518537,9.097636,0.758621,2009.0,2.040115
49,,Bleach: Sennen Kessen-hen,41467,0.483582,9.048079,0.982759,2022.0,2.031586
45,,One Piece,21,0.487591,8.686696,0.586207,1999.0,1.929409
7,,Naruto: Shippuuden,1735,0.553618,8.257899,0.724138,2007.0,1.907615
36,,Yuu☆Yuu☆Hakusho,392,0.505619,8.448490,0.465517,1992.0,1.885820
67,,Jigokuraku,46569,0.476893,8.224549,1.000000,2023.0,1.880862
9,,D.Gray-man,1482,0.552713,8.009810,0.706897,2006.0,1.860664
73,,Magi: The Kingdom of Magic,18115,0.475000,8.212752,0.827586,2013.0,1.860304
53,,Hunter x Hunter: Original Video Animation,137,0.481455,8.258539,0.637931,2002.0,1.853647


In [ ]:
df = get_recommendations4(title='Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
52,,Shigatsu wa Kimi no Uso,23273,0.465837,8.646914,0.808511,2014.0,1.930773
99,,Vivy: Fluorite Eye's Song,46095,0.447036,8.394183,0.957447,2021.0,1.888330
18,,Lycoris Recoil,50709,0.492501,8.183950,0.978723,2022.0,1.881259
83,,Kidou Senshi Gundam: Tekketsu no Orphans 2nd Season,33051,0.450011,8.188491,0.851064,2016.0,1.842542
9,,Plastic Memories,27775,0.518499,7.904120,0.829787,2015.0,1.832374
29,,Senki Zesshou Symphogear XV,32843,0.482099,7.975510,0.914894,2019.0,1.830804
32,,Carole & Tuesday,37435,0.481744,7.851340,0.914894,2019.0,1.808229
85,,Kidou Senshi Gundam 00,2581,0.449888,8.084751,0.659574,2007.0,1.804642
63,,Kidou Senshi Gundam 00 Second Season,3927,0.455764,8.048096,0.680851,2008.0,1.803874
8,,Texhnolyze,26,0.521140,7.717234,0.574468,2003.0,1.774867


In [ ]:
get_recommendations4(title='Darling in the FranXX', n=24)

,Image URL,Name,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
52,https://cdn.myanimelist.net/images/anime/3/671...,Shigatsu wa Kimi no Uso,23273,0.465837,8.646914,0.808511,2014.0,1.930773
99,https://cdn.myanimelist.net/images/anime/1637/...,Vivy: Fluorite Eye's Song,46095,0.447036,8.394183,0.957447,2021.0,1.888330
18,https://cdn.myanimelist.net/images/anime/1392/...,Lycoris Recoil,50709,0.492501,8.183950,0.978723,2022.0,1.881259
83,https://cdn.myanimelist.net/images/anime/6/808...,Kidou Senshi Gundam: Tekketsu no Orphans 2nd S...,33051,0.450011,8.188491,0.851064,2016.0,1.842542
9,https://cdn.myanimelist.net/images/anime/4/727...,Plastic Memories,27775,0.518499,7.904120,0.829787,2015.0,1.832374
29,https://cdn.myanimelist.net/images/anime/1899/...,Senki Zesshou Symphogear XV,32843,0.482099,7.975510,0.914894,2019.0,1.830804
32,https://cdn.myanimelist.net/images/anime/1611/...,Carole & Tuesday,37435,0.481744,7.851340,0.914894,2019.0,1.808229
85,https://cdn.myanimelist.net/images/anime/3/132...,Kidou Senshi Gundam 00,2581,0.449888,8.084751,0.659574,2007.0,1.804642
63,https://cdn.myanimelist.net/images/anime/9/127...,Kidou Senshi Gundam 00 Second Season,3927,0.455764,8.048096,0.680851,2008.0,1.803874
8,https://cdn.myanimelist.net/images/anime/1027/...,Texhnolyze,26,0.521140,7.717234,0.574468,2003.0,1.774867


#### Conclusion:
Effect of Fine-Tuned Feature Weights and Recency in `get_recommendations4`

The updated results for *`Darling in the FranXX`* using `get_recommendations4()` demonstrate the impact of **fine-tuning feature similarity weights** and applying a **recency-aware scoring formula**.

#### 🔍 Key Improvements:

* The top results now prioritize **recent anime** (e.g., *`Lycoris Recoil`* (2022), *`Vivy`* (2021), *`Plastic Memories`* (2015), *`SSSS.Dynazenon`* (2021)).
* Older but highly relevant series (like *`Texhnolyze`* or *`Gundam titles`*) are still present, but appear lower than in earlier versions — showing a **more balanced mix of new and classic titles**.
* The final scores reflect this trade-off: titles with moderate similarity but high **recency and rating** (like *`Shigatsu wa Kimi no Uso`* or *`Vivy`*) outperform older titles with similar genres but less recency impact.


By **tuning the similarity matrix** (`combined_sim2`) and adjusting the **final score formula to incorporate recency**, the system produces **more modern, user-relevant recommendations** while maintaining thematic consistency. This approach increases discovery of newer anime without disregarding high-quality classics — striking a thoughtful balance between relevance and freshness.

### Lets adjust the weifhts one more time to get more satidying results

### **Combine Similarities Using Fine-Tuned Feature Weights**


This cell creates a new **hybrid content similarity matrix** (`combined_sim3`) by assigning **custom weights** to each encoded feature, based on assumed or collected **user preferences**:

* **Genres**: `30%` — important for thematic alignment
* **Synopsis**: `35%` — most descriptive of plot and tone
* **Type**: `15%` — format (e.g., TV vs. Movie) moderately impacts relevance
* **Studios**: `10%` — aesthetic/style influence
* **Episodes**: `5%` — minor but helpful for pacing or length matching
* **Source**: `5%` — light effect (e.g., manga vs. original)

These weights reflect the relative importance of each feature in shaping user preferences, resulting in a **more personalized similarity matrix**.

In [ ]:
# Fine-tuned weights based on user preferences
weights = [0.30, 0.35, 0.15, 0.10, 0.05, 0.05]

# Combine similarities
combined_sim3 = (weights[0] * genres_sim +  # Genres: 0.30
                weights[1] * synopsis_sim +  # Synopsis: 0.35
                weights[2] * type_sim +      # Type: 0.15
                weights[3] * studios_sim +   # Studios: 0.10
                weights[4] * episodes_sim +  # Episodes: 0.05
                weights[5] * source_sim  # Source: 0.05
                )

#### **Create Normalized Recency Score from Release Year**

This utility function creates a **`recency_score`** ranging from `0` to `1` based on the anime's release year. It helps prioritize newer titles during recommendation scoring.

#### **Methodology**:

* Reads the anime's **`release_year`** column.
* Fills missing years with the **minimum known year** (to avoid boosting incomplete data).
* Applies **min-max normalization**:

  $$\text{recency_score} = \frac{(\text{year} - \text{min})}{(\text{max}-\text{ min})}$$
* Returns the same DataFrame with a new column: **`recency_score`**

This allows the system to **boost newer anime titles** when computing the final recommendation score.

In [ ]:
def _apply_recency_boost(df):
    """
    Uses the numeric 'release_year' column to create a 0–1 scaled 'recency_score'.
    """
    # Use 'release_year' directly
    df['year'] = df['release_year']

    # Fallback for missing years (fill with oldest year)
    df['year'] = df['year'].fillna(df['year'].min())

    # Scale to [0, 1]
    min_year = df['year'].min()
    max_year = df['year'].max()
    year_range = max_year - min_year if max_year != min_year else 1

    df['recency_score'] = (df['year'] - min_year) / year_range

    return df


#### **Final Hybrid Recommendation Function with Fine-Tuned Weights and Recency Boost**

* `get_recommendations101()` delivers anime recommendations by combining:

  1. **Fine-tuned similarity scores** from `combined_sim3`, based on weighted feature similarity (genres, synopsis, type, etc.)
  2. **Weighted user ratings** for quality assessment
  3. **Recency scoring** to boost newer releases

* **Methodology**:

  * Finds the input title in `anime_df`.
  * Retrieves the most similar anime using `combined_sim3`, which includes custom-tuned feature weights.
  * Applies `_apply_recency_boost()` to normalize each anime’s release year into a `recency_score` (0–1).
  * Calculates a composite score:

  $$
  \text{final_score} = \left[ \text{rating} \cdot (1 - w_s) + \text{similarity} \cdot w_s \right] \cdot (1 - w_r) + \text{recency} \cdot w_r
  $$

  Where:

  * $w_s$: similarity weight (default 0.85)
  * $w_r$: recency weight (default 0.20)

* **Output**:

  Returns the **top N recommendations**, sorted by the `final_score`, including:

  * Anime name, type, image, rating, similarity, recency score, and release year

In [ ]:
def get_recommendations101(title, n=10, similarity_weight=0.85, recency_weight=0.2):
    # Check if title exists and get its label index
    matching_animes = anime_df[anime_df['Name'] == title]
    if matching_animes.empty:
        raise ValueError(f"Anime '{title}' not found in the database.")
    label = matching_animes.index[0]

    pos = anime_df.index.get_loc(label)
    sim_scores = list(enumerate(combined_sim3[pos]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    max_recommendations = min(52, len(anime_df) - 1)
    top_sim = sim_scores[1:max_recommendations + 1]
    sim_indices = [i for i, _ in top_sim]
    sim_scores_top = [score for _, score in top_sim]

    recommended_animes = anime_df.iloc[sim_indices][
        ['Name', 'anime_id', 'weighted_rating', 'Image URL', 'Type', 'Genres', 'Score', 'release_year']
    ].copy()

    recommended_animes = _apply_recency_boost(recommended_animes)
    qualified_animes = recommended_animes.copy()
    qualified_animes.reset_index(drop=True, inplace=True)

    similarity_df = pd.DataFrame({
        "anime_id": anime_df.iloc[sim_indices]["anime_id"].values,
        "similarity_score": sim_scores_top
    })

    qualified_animes = qualified_animes.merge(similarity_df, on="anime_id", how="left")
    qualified_animes["similarity_score"] = qualified_animes["similarity_score"].fillna(
        qualified_animes["similarity_score"].min()
    )

    rating_weight = 1 - similarity_weight
    if rating_weight < 0:
        raise ValueError("The sum of similarity_weight and recency_weight must be less than or equal to 1.")

    qualified_animes['final_score'] = (
        rating_weight * qualified_animes['weighted_rating'] +
        similarity_weight * qualified_animes['similarity_score'])* (1-recency_weight) + (recency_weight * qualified_animes['recency_score'])

    return qualified_animes.sort_values('final_score', ascending=False).head(n)[
        ['Image URL', 'Name','Type', 'anime_id',  'similarity_score', 'weighted_rating', 'recency_score', 'release_year', 'final_score']
    ]

* **Lets do some tests**

In [ ]:
df = get_recommendations101(title='Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Type,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
37,,86 Part 2,TV,48569,0.474157,8.692836,0.977778,2021.0,1.561122
32,,Shigatsu wa Kimi no Uso,TV,23273,0.476193,8.646914,0.822222,2014.0,1.525885
30,,86,TV,41457,0.477333,8.270491,0.977778,2021.0,1.512601
10,,Plastic Memories,TV,27775,0.511800,7.904120,0.844444,2015.0,1.465407
38,,Kidou Senshi Gundam 00 Second Season,TV,3927,0.474065,8.048096,0.688889,2008.0,1.425914
3,,Kiznaiver,TV,31798,0.520255,7.374965,0.866667,2016.0,1.412103
26,,SSSS.Dynazenon,TV,40870,0.478243,7.375668,0.977778,2021.0,1.405841
8,,Soukyuu no Fafner: Dead Aggressor - Exodus Part 2,TV,30549,0.515709,7.309665,0.844444,2015.0,1.396731
43,,Senki Zesshou Symphogear AXZ,TV,32836,0.472047,7.472081,0.888889,2017.0,1.395420
5,,Guilty Crown,TV,10793,0.517757,7.417078,0.755556,2011.0,1.393235


In [ ]:
df = get_recommendations101(title='Jujutsu Kaisen', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Type,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
2,,Chainsaw Man,TV,44511,0.553409,8.584003,0.974359,2022.0,1.601270
19,,Bleach: Sennen Kessen-hen,TV,41467,0.462662,9.048079,0.974359,2022.0,1.595251
17,,Shingeki no Kyojin: The Final Season,TV,40028,0.474939,8.796520,0.923077,2020.0,1.563156
4,,Jigokuraku,TV,46569,0.551048,8.224549,1.000000,2023.0,1.561658
36,,Vinland Saga Season 2,TV,49387,0.452953,8.771356,1.000000,2023.0,1.560571
34,,Kimetsu no Yaiba: Yuukaku-hen,TV,47778,0.453753,8.794506,0.948718,2021.0,1.553637
7,,Kimetsu no Yaiba,TV,38000,0.506242,8.498085,0.897436,2019.0,1.543502
27,,Kimetsu no Yaiba: Katanakaji no Sato-hen,TV,51019,0.457070,8.467293,1.000000,2023.0,1.526882
25,,Kimetsu no Yaiba: Mugen Ressha-hen,TV,49926,0.457433,8.380549,0.948718,2021.0,1.506464
16,,Dorohedoro,TV,38668,0.481031,8.048611,0.923077,2020.0,1.477550


In [ ]:
df = get_recommendations101(title='Naruto: Shippuuden', n=10)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Type,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
3,,Bleach: Sennen Kessen-hen,TV,41467,0.608061,9.048079,0.976744,2022.0,1.694600
15,,Hunter x Hunter (2011),TV,11061,0.554940,9.037173,0.720930,2011.0,1.606006
0,,Naruto,TV,20,0.774972,7.988501,0.511628,2002.0,1.587927
38,,Fullmetal Alchemist: Brotherhood,TV,5114,0.517224,9.097636,0.674419,2009.0,1.578312
17,,Black Clover,TV,34572,0.554523,8.136070,0.860465,2017.0,1.525497
8,,Akatsuki no Yona,TV,25013,0.566378,8.022766,0.790698,2014.0,1.506009
2,,Bleach,TV,269,0.652481,7.917476,0.558140,2004.0,1.505412
29,,Magi: The Kingdom of Magic,TV,18115,0.521890,8.212752,0.767442,2013.0,1.493904
13,,Dragon Quest: Dai no Daibouken (2020),TV,40906,0.555825,7.643075,0.930233,2020.0,1.481176
41,,One Piece,TV,21,0.514408,8.686696,0.441860,1999.0,1.480573


In [ ]:
get_recommendations101(title='Naruto: Shippuuden', n=24)

,Image URL,Name,Type,anime_id,similarity_score,weighted_rating,recency_score,release_year,final_score
3,https://cdn.myanimelist.net/images/anime/1908/...,Bleach: Sennen Kessen-hen,TV,41467,0.608061,9.048079,0.976744,2022.0,1.694600
15,https://cdn.myanimelist.net/images/anime/1337/...,Hunter x Hunter (2011),TV,11061,0.554940,9.037173,0.720930,2011.0,1.606006
0,https://cdn.myanimelist.net/images/anime/13/17...,Naruto,TV,20,0.774972,7.988501,0.511628,2002.0,1.587927
38,https://cdn.myanimelist.net/images/anime/1208/...,Fullmetal Alchemist: Brotherhood,TV,5114,0.517224,9.097636,0.674419,2009.0,1.578312
17,https://cdn.myanimelist.net/images/anime/2/883...,Black Clover,TV,34572,0.554523,8.136070,0.860465,2017.0,1.525497
8,https://cdn.myanimelist.net/images/anime/9/642...,Akatsuki no Yona,TV,25013,0.566378,8.022766,0.790698,2014.0,1.506009
2,https://cdn.myanimelist.net/images/anime/3/404...,Bleach,TV,269,0.652481,7.917476,0.558140,2004.0,1.505412
29,https://cdn.myanimelist.net/images/anime/13/55...,Magi: The Kingdom of Magic,TV,18115,0.521890,8.212752,0.767442,2013.0,1.493904
13,https://cdn.myanimelist.net/images/anime/1499/...,Dragon Quest: Dai no Daibouken (2020),TV,40906,0.555825,7.643075,0.930233,2020.0,1.481176
41,https://cdn.myanimelist.net/images/anime/6/732...,One Piece,TV,21,0.514408,8.686696,0.441860,1999.0,1.480573


#### **Final Analysis of `get_recommendations101()` for *Naruto: Shippuuden***

The output of `get_recommendations101()` demonstrates the most balanced and refined results compared to previous recommendation versions. It effectively integrates:

* **Semantic relevance** through a finely tuned similarity matrix (`combined_sim3`)
* **Quality assurance** via weighted user ratings
* **Freshness bias** using a normalized recency score

---

### **Why These Are the Best Results Yet**

####  **1. Relevance & Genre Alignment**

The top recommendations (e.g., *Bleach: Sennen Kessen-hen*, *Hunter x Hunter*, *Naruto*, *Fullmetal Alchemist: Brotherhood*) are **deeply thematically aligned** with *Naruto: Shippuuden*. They all:

* Fall within the **shounen**, **action**, **fantasy**, or **adventure** genres
* Include strong narratives, ensemble casts, and character development arcs

This confirms that the **custom similarity weights** are functioning well — giving priority to genres and plot structure (e.g., `synopsis_weight = 0.35`, `genre_weight = 0.30`).


#### **2. Quality Scores Still Matter**

Highly rated titles (e.g., *Hunter x Hunter (2011)*, *Fullmetal Alchemist: Brotherhood*) appear prominently — **despite being older** — thanks to their exceptional user scores.

This proves the system does **not over-prioritize recency**, maintaining a **strong quality bias** through `weighted_rating`.


#### **3. Smart Recency Boost**

Newer anime like:

* *Bleach: Sennen Kessen-hen* (2022)
* *Black Clover* (2017)
* *Dragon Quest: Dai no Daibouken* (2020)
  appear near the top of the list, showing that **modern titles are correctly being elevated** in the ranking — without overshadowing the all-time greats.

This affirms that the **recency weight (0.2)** strikes the right balance between **discovery** and **classic appeal**.

---

### **Conclusion: Best Results So Far**

Compared to previous versions (`v2`, `v3`, and `v4`):

* The results here are **more current**, **more contextually aligned**, and **less biased toward only legacy titles**
* The hybrid scoring formula allows **older masterpieces and modern hits to coexist fairly**
* The tuning of content feature weights in `combined_sim3` allows the system to understand **what truly matters to the viewer**


#### **Lets put them into a class**

The `AnimeRecommender` class is a content-based hybrid recommendation system that suggests similar anime using genre, synopsis, type, studio, episode, and source features. It preprocesses data, computes weighted similarities, and enhances results with a recency boost to favor newer titles. It supports both item-based and user-based recommendations with optimized performance using sparse matrices.

In [ ]:


class AnimeRecommender:
    def __init__(self, anime_df, ratings_df):
        # Data loading and type optimization
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self._preprocess_data()
        self._create_feature_matrices()

        # Similarity weights
        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        """Data cleaning and feature engineering with memory optimization"""
        # Handle missing values
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)

        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')

        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)

        self.anime_df['release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])

        # Weighted rating
        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )


    def _create_feature_matrices(self):
        """Create sparse feature matrices for fast similarity calculations"""

        # Multi-label genres and studios
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)


        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        # TF-IDF for synopsis
        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'])
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        """Clean text data by lowercasing and removing punctuation.

        Args:
            text_series: Series of text to clean.

        Returns:
            Cleaned text Series.
        """
        text = text_series.str.lower()
        text = text.str.replace(r'[^\w\s]', '', regex=True)
        return text

    def _calculate_anime_similarity(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        """Calculate weighted similarity for a single anime using sparse operations.

        Args:
            idx: Index of the target anime.

        Returns:
            Tuple of similarity scores and corresponding indices.
        """
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        top_indices = sim_scores[:, 1].argsort()[-(52):-1][::-1]
        sim_scores_top = sim_scores[top_indices, 1]
        return sim_scores_top, top_indices

    def _apply_recency_boost(self, df):
        df['year'] = df['release_year']
        df['year'] = df['year'].fillna(df['year'].min())
        min_year = df['year'].min()
        max_year = df['year'].max()
        year_range = max_year - min_year if max_year != min_year else 1
        df['recency_score'] = (df['year'] - min_year) / year_range
        # current_year = 2025.0
        # age = current_year - df['year']
        # df['recency_score'] = 1 / (1 + np.exp(-(10 - age) / 2))
        return df

    def get_anime_recommendations(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        # Compute similarity vectors
        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        # Fetch recommended anime
        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id',  'Type', 'Genres', 'Score', 'weighted_rating', 'release_year']
        ].copy()
        recommended_animes = self._apply_recency_boost(recommended_animes)
        recommended_animes['similarity_score'] = sim_scores_top

        # Compute final score
        recommended_animes['final_score'] = (
            (1-similarity_weight) * recommended_animes['weighted_rating'] +
            similarity_weight * recommended_animes['similarity_score'])* (1-recency_weight) + (recency_weight * recommended_animes['recency_score'])

        return recommended_animes.sort_values('final_score', ascending=False).head(n)

* create an inastance from it

In [ ]:
recommender = AnimeRecommender(anime_df, ratings_df)

Lets test the recommendations

In [ ]:
df = recommender.get_anime_recommendations('Kimetsu no Yaiba', n=10)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,release_year,year,recency_score,similarity_score,final_score
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.606117,1.657244
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.642217,1.652783
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.604693,1.606601
16236,,Jujutsu Kaisen,40748,TV,"[Action, Award Winning, Fantasy]",8.64,8.637287,2020.0,2020.0,0.923077,0.506242,1.565335
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.454961,1.534326
16060,,Kimetsu no Yaiba Movie: Mugen Ressha-hen,40456,Movie,"[Action, Fantasy]",8.62,8.615747,2020.0,2020.0,0.923077,0.453053,1.526581
6589,,Fate/Zero 2nd Season,11741,TV,"[Action, Fantasy, Supernatural]",8.55,8.544452,2012.0,2012.0,0.717949,0.456234,1.479163
14835,,Toaru Kagaku no Railgun T,38481,TV,"[Action, Fantasy, Sci-Fi]",8.17,8.134640,2020.0,2020.0,0.923077,0.450000,1.466772
9821,,Fate/stay night: Unlimited Blade Works 2nd Season,28701,TV,"[Action, Fantasy, Supernatural]",8.32,8.313739,2015.0,2015.0,0.794872,0.454354,1.465584
10527,,Noragami Aragoto,30503,TV,"[Action, Fantasy]",8.16,8.156397,2015.0,2015.0,0.794872,0.449720,1.443551


In [ ]:
df = recommender.get_anime_recommendations('Jujutsu Kaisen', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,release_year,year,recency_score,similarity_score,final_score
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.553409,1.601270
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[Action, Adventure, Fantasy]",9.07,9.048079,2022.0,2022.0,0.974359,0.462662,1.595251
15822,,Shingeki no Kyojin: The Final Season,40028,TV,"[Action, Drama]",8.80,8.796520,2020.0,2020.0,0.923077,0.474939,1.563156
19600,,Jigokuraku,46569,TV,"[Action, Adventure, Fantasy]",8.26,8.224549,2023.0,2023.0,1.000000,0.551048,1.561658
21303,,Vinland Saga Season 2,49387,TV,"[Action, Adventure, Drama]",8.81,8.771356,2023.0,2023.0,1.000000,0.452953,1.560571
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.453753,1.553637
14539,,Kimetsu no Yaiba,38000,TV,"[Action, Award Winning, Fantasy]",8.50,8.498085,2019.0,2019.0,0.897436,0.506242,1.543502
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.457070,1.526882
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.457433,1.506464
14942,,Dorohedoro,38668,TV,"[Action, Comedy, Fantasy, Horror]",8.06,8.048611,2020.0,2020.0,0.923077,0.481031,1.477550


In [ ]:
df = recommender.get_anime_recommendations('Darling in the FranXX', n=10)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,release_year,year,recency_score,similarity_score,final_score
20964,,86 Part 2,48569,TV,"[Action, Drama, Sci-Fi]",8.71,8.692836,2021.0,2021.0,0.977778,0.474157,1.561122
8855,,Shigatsu wa Kimi no Uso,23273,TV,"[Drama, Romance]",8.65,8.646914,2014.0,2014.0,0.822222,0.476193,1.525885
16608,,86,41457,TV,"[Action, Drama, Sci-Fi]",8.28,8.270491,2021.0,2021.0,0.977778,0.477333,1.512601
9604,,Plastic Memories,27775,TV,"[Drama, Romance, Sci-Fi]",7.91,7.904120,2015.0,2015.0,0.844444,0.511800,1.465407
3375,,Kidou Senshi Gundam 00 Second Season,3927,TV,"[Action, Drama, Sci-Fi]",8.08,8.048096,2008.0,2008.0,0.688889,0.474065,1.425914
11072,,Kiznaiver,31798,TV,"[Drama, Romance, Sci-Fi]",7.38,7.374965,2016.0,2016.0,0.866667,0.520255,1.412103
16309,,SSSS.Dynazenon,40870,TV,"[Action, Sci-Fi]",7.42,7.375668,2021.0,2021.0,0.977778,0.478243,1.405841
10540,,Soukyuu no Fafner: Dead Aggressor - Exodus Part 2,30549,TV,"[Action, Drama, Sci-Fi]",7.61,7.309665,2015.0,2015.0,0.844444,0.515709,1.396731
11508,,Senki Zesshou Symphogear AXZ,32836,TV,"[Action, Sci-Fi]",7.60,7.472081,2017.0,2017.0,0.888889,0.472047,1.395420
6358,,Guilty Crown,10793,TV,"[Action, Drama, Sci-Fi]",7.42,7.417078,2011.0,2011.0,0.755556,0.517757,1.393235


Despite offering functional recommendations, the earlier version of the AnimeRecommender class suffered from a notable limitation in how it processed anime synopses. Specifically, the system applied only basic text normalization—lowercasing and punctuation removal—before generating TF-IDF vectors. This naive approach overlooked crucial linguistic elements such as synonyms, verb/noun variations, and context-specific word usage. As a result, the system often failed to capture the deeper semantic similarity between anime plots, especially when two shows shared core themes but used different vocabulary. This limitation diluted the effectiveness of synopsis-based matching, which is arguably one of the most important content features. To address these shortcomings, the updated class integrates a full NLP pipeline—leveraging tokenization, stopword removal, POS tagging, and lemmatization—to produce cleaner and more semantically rich representations, enabling more accurate and meaningful recommendations.

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')


class AnimeRecommender:
    def __init__(self, anime_df, ratings_df):
        """Initialize the recommender with anime and ratings data."""
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        # Initialize NLP tools
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self._preprocess_data()
        self._create_feature_matrices()

        # Similarity weights
        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        """Data cleaning and feature engineering with memory optimization."""
        # Handle missing values
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)

        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')

        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)

        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])

        # Weighted rating
        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        """Create sparse feature matrices for fast similarity calculations."""
        # Multi-label genres and studios
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        # TF-IDF for synopsis with enhanced cleaning
        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    @staticmethod
    def get_wordnet_pos(tag):
        """Map NLTK POS tag to WordNet POS tag for lemmatization.

        Args:
            tag: NLTK POS tag (e.g., 'NN', 'VB').

        Returns:
            WordNet POS tag ('n', 'v', 'a', 'r').
        """
        if tag.startswith('J'):
            return 'a'  # adjective
        elif tag.startswith('V'):
            return 'v'  # verb
        elif tag.startswith('N'):
            return 'n'  # noun
        elif tag.startswith('R'):
            return 'r'  # adverb
        else:
            return 'n'  # default to noun

    def _lemmatize_text(self, text: str) -> str:
        """Lemmatize and clean a single text string.

        Args:
            text: Input text to clean.

        Returns:
            Cleaned and lemmatized text as a string.
        """
        text = text.lower()
        tokens = word_tokenize(text)
        tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
        cleaned_tokens = []
        for token, tag in tagged_tokens:
            if token not in string.punctuation and token not in self.stop_words:
                wordnet_pos = self.get_wordnet_pos(tag)
                lemma = self.lemmatizer.lemmatize(token, pos=wordnet_pos)
                cleaned_tokens.append(lemma)
        return ' '.join(cleaned_tokens)

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        """Clean text data by lowercasing, tokenizing, removing punctuation and stopwords, and lemmatizing.

        Args:
            text_series: Series of text to clean.

        Returns:
            Cleaned text Series.
        """
        return text_series.apply(self._lemmatize_text)

    def _calculate_anime_similarity(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        """Calculate weighted similarity for a single anime using sparse operations.

        Args:
            idx: Index of the target anime.

        Returns:
            Tuple of similarity scores and corresponding indices.
        """
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        top_indices = sim_scores[:, 1].argsort()[-(52):-1][::-1]
        sim_scores_top = sim_scores[top_indices, 1]
        return sim_scores_top, top_indices

    def _apply_recency_boost(self, df):
        """Apply a recency boost based on release year."""
        df['year'] = df['Release_year']
        df['year'] = df['year'].fillna(df['year'].min())
        min_year = df['year'].min()
        max_year = df['year'].max()
        year_range = max_year - min_year if max_year != min_year else 1
        df['recency_score'] = (df['year'] - min_year) / year_range
        return df

    def get_anime_recommendations(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        """Generate anime recommendations based on a given title."""
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year']
        ].copy()
        recommended_animes = self._apply_recency_boost(recommended_animes)
        recommended_animes['similarity_score'] = sim_scores_top

        recommended_animes['final_score'] = (
            (1 - similarity_weight) * recommended_animes['weighted_rating'] +
            similarity_weight * recommended_animes['similarity_score']
        ) * (1 - recency_weight) + (recency_weight * recommended_animes['recency_score'])

        return recommended_animes.sort_values('final_score', ascending=False).head(n)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


* Lets make and instance of the new class then test it...

In [ ]:
recommender = AnimeRecommender(anime_df, ratings_df)
df = recommender.get_anime_recommendations('Bakemonogatari', n=10)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
12861,,Owarimonogatari 2nd Season,35247,TV,"[Comedy, Mystery, Supernatural]",8.88,8.856249,2017.0,2017.0,0.76,0.642811,1.651861
7556,,Monogatari Series: Second Season,17074,TV,"[Comedy, Mystery, Romance, Supernatural]",8.77,8.757932,2013.0,2013.0,0.60,0.652100,1.614380
10808,,Owarimonogatari,31181,TV,"[Comedy, Mystery, Supernatural]",8.45,8.434069,2015.0,2015.0,0.68,0.606948,1.560813
11052,,Kizumonogatari III: Reiketsu-hen,31758,Movie,"[Action, Mystery, Supernatural]",8.79,8.773543,2017.0,2017.0,0.76,0.475048,1.527858
6553,,Nisemonogatari,11597,TV,"[Comedy, Mystery, Supernatural, Ecchi]",8.14,8.132676,2012.0,2012.0,0.56,0.636405,1.520676
7281,,Nekomonogatari: Kuro,15689,TV,"[Comedy, Romance, Supernatural, Ecchi]",7.93,7.921766,2012.0,2012.0,0.56,0.634826,1.494293
13907,,Zoku Owarimonogatari,36999,Movie,"[Comedy, Mystery, Supernatural]",8.45,8.414520,2018.0,2018.0,0.80,0.468357,1.488225
14179,,Seishun Buta Yarou wa Bunny Girl Senpai no Yume wo Minai,37450,TV,"[Drama, Romance, Supernatural]",8.24,8.236870,2018.0,2018.0,0.80,0.476443,1.472405
9660,,Tsukimonogatari,28025,TV,"[Comedy, Mystery, Supernatural, Ecchi]",8.09,8.077023,2014.0,2014.0,0.64,0.550759,1.471759
5670,,Kizumonogatari I: Tekketsu-hen,9260,Movie,"[Action, Mystery, Supernatural]",8.37,8.357224,2016.0,2016.0,0.72,0.473680,1.468969


In [ ]:
df = recommender.get_anime_recommendations('Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
20964,,86 Part 2,48569,TV,"[Action, Drama, Sci-Fi]",8.71,8.692836,2021.0,2021.0,0.977778,0.476150,1.562478
8855,,Shigatsu wa Kimi no Uso,23273,TV,"[Drama, Romance]",8.65,8.646914,2014.0,2014.0,0.822222,0.479830,1.528358
16608,,86,41457,TV,"[Action, Drama, Sci-Fi]",8.28,8.270491,2021.0,2021.0,0.977778,0.477433,1.512669
21948,,Lycoris Recoil,50709,TV,[Action],8.20,8.183950,2022.0,2022.0,1.000000,0.472867,1.503623
9604,,Plastic Memories,27775,TV,"[Drama, Romance, Sci-Fi]",7.91,7.904120,2015.0,2015.0,0.844444,0.517298,1.469146
3375,,Kidou Senshi Gundam 00 Second Season,3927,TV,"[Action, Drama, Sci-Fi]",8.08,8.048096,2008.0,2008.0,0.688889,0.480428,1.430240
11072,,Kiznaiver,31798,TV,"[Drama, Romance, Sci-Fi]",7.38,7.374965,2016.0,2016.0,0.866667,0.522201,1.413426
16309,,SSSS.Dynazenon,40870,TV,"[Action, Sci-Fi]",7.42,7.375668,2021.0,2021.0,0.977778,0.479599,1.406763
10540,,Soukyuu no Fafner: Dead Aggressor - Exodus Part 2,30549,TV,"[Action, Drama, Sci-Fi]",7.61,7.309665,2015.0,2015.0,0.844444,0.515613,1.396666
16,,Texhnolyze,26,TV,"[Action, Drama, Sci-Fi]",7.76,7.717234,2003.0,2003.0,0.577778,0.517278,1.393373


In [ ]:
df = recommender.get_anime_recommendations('One Piece', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
14699,,One Piece Movie 14: Stampede,38234,Movie,"[Action, Adventure, Fantasy]",8.22,8.190025,2019.0,2019.0,0.868421,0.578450,1.549833
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[Action, Adventure, Fantasy]",7.74,7.643075,2020.0,2020.0,0.894737,0.615499,1.514656
13409,,One Piece: Episode of East Blue - Luffy to 4-nin no Nakama no Daibouken,36215,Special,"[Action, Adventure, Fantasy]",7.88,7.758384,2017.0,2017.0,0.815789,0.600029,1.502183
6823,,One Piece Film: Z,12859,Movie,"[Action, Adventure, Fantasy]",8.14,8.121859,2012.0,2012.0,0.684211,0.574520,1.502139
17060,,Heion Sedai no Idaten-tachi,42625,TV,"[Action, Adventure, Fantasy]",7.64,7.609725,2021.0,2021.0,0.921053,0.558974,1.477480
10865,,Drifters,31339,TV,"[Action, Adventure, Comedy, Fantasy]",7.90,7.889898,2016.0,2016.0,0.789474,0.520367,1.458532
3514,,One Piece Film: Strong World,4155,Movie,"[Action, Adventure, Fantasy]",8.08,8.060354,2009.0,2009.0,0.605263,0.542080,1.456910
1574,,Naruto: Shippuuden,1735,TV,"[Action, Adventure, Fantasy]",8.26,8.257899,2007.0,2007.0,0.552632,0.519085,1.454452
11043,,Magi: Sinbad no Bouken (TV),31741,TV,"[Action, Adventure, Fantasy]",7.85,7.839802,2016.0,2016.0,0.789474,0.522789,1.454167
7119,,Magi: The Labyrinth of Magic,14513,TV,"[Action, Adventure, Fantasy]",8.02,8.014632,2012.0,2012.0,0.684211,0.517040,1.450185


In [ ]:
df = recommender.get_anime_recommendations('Sankarea', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
7481,,Kami nomi zo Shiru Sekai: Megami-hen,16706,TV,"[Comedy, Romance, Supernatural]",8.00,7.985126,2013.0,2013.0,0.756098,0.488362,1.441521
6356,,Kore wa Zombie desu ka? of the Dead,10790,TV,"[Action, Comedy, Supernatural, Ecchi]",7.49,7.480967,2012.0,2012.0,0.731707,0.557525,1.423175
6042,,Kami nomi zo Shiru Sekai II,10080,TV,"[Comedy, Romance, Supernatural]",7.88,7.868894,2011.0,2011.0,0.707317,0.484432,1.415144
10393,,Prison School,30240,TV,"[Comedy, Romance, Ecchi]",7.61,7.606261,2015.0,2015.0,0.804878,0.491384,1.407868
9339,,Junjou Romantica 3,25649,TV,"[Boys Love, Comedy, Drama, Romance]",7.62,7.585614,2015.0,2015.0,0.804878,0.493986,1.407160
11346,,Sakamoto desu ga?,32542,TV,[Comedy],7.55,7.544551,2016.0,2016.0,0.829268,0.488381,1.403299
6868,,Sankarea OVA,13055,OVA,"[Comedy, Horror, Romance, Supernatural, Ecchi]",7.20,7.176518,2012.0,2012.0,0.731707,0.578102,1.400633
101,,Full Moon wo Sagashite,122,TV,"[Comedy, Drama, Romance, Supernatural]",7.94,7.883657,2002.0,2002.0,0.487805,0.507293,1.388559
5400,,Kami nomi zo Shiru Sekai,8525,TV,"[Comedy, Romance, Supernatural]",7.66,7.653154,2010.0,2010.0,0.682927,0.489389,1.387748
5528,,Kore wa Zombie desu ka?,8841,TV,"[Action, Comedy, Supernatural, Ecchi]",7.35,7.344986,2011.0,2011.0,0.707317,0.534388,1.386245


In [ ]:
df = recommender.get_anime_recommendations('Kimetsu no Yaiba', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.675826,1.675637
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.627257,1.671619
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.627586,1.622168
16236,,Jujutsu Kaisen,40748,TV,"[Action, Award Winning, Fantasy]",8.64,8.637287,2020.0,2020.0,0.923077,0.510968,1.568548
16060,,Kimetsu no Yaiba Movie: Mugen Ressha-hen,40456,Movie,"[Action, Fantasy]",8.62,8.615747,2020.0,2020.0,0.923077,0.493573,1.554134
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.456532,1.535394
6589,,Fate/Zero 2nd Season,11741,TV,"[Action, Fantasy, Supernatural]",8.55,8.544452,2012.0,2012.0,0.717949,0.460878,1.482321
14835,,Toaru Kagaku no Railgun T,38481,TV,"[Action, Fantasy, Sci-Fi]",8.17,8.134640,2020.0,2020.0,0.923077,0.454360,1.469737
9821,,Fate/stay night: Unlimited Blade Works 2nd Season,28701,TV,"[Action, Fantasy, Supernatural]",8.32,8.313739,2015.0,2015.0,0.794872,0.453259,1.464839
10527,,Noragami Aragoto,30503,TV,"[Action, Fantasy]",8.16,8.156397,2015.0,2015.0,0.794872,0.449990,1.443735


In [ ]:
df = recommender.get_anime_recommendations('Jujutsu Kaisen', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.550663,1.599403
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[Action, Adventure, Fantasy]",9.07,9.048079,2022.0,2022.0,0.974359,0.462249,1.594971
15822,,Shingeki no Kyojin: The Final Season,40028,TV,"[Action, Drama]",8.80,8.796520,2020.0,2020.0,0.923077,0.478012,1.565246
21303,,Vinland Saga Season 2,49387,TV,"[Action, Adventure, Drama]",8.81,8.771356,2023.0,2023.0,1.000000,0.455136,1.562055
19600,,Jigokuraku,46569,TV,"[Action, Adventure, Fantasy]",8.26,8.224549,2023.0,2023.0,1.000000,0.551210,1.561769
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.453632,1.553554
14539,,Kimetsu no Yaiba,38000,TV,"[Action, Award Winning, Fantasy]",8.50,8.498085,2019.0,2019.0,0.897436,0.510968,1.546716
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.457350,1.527073
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.458439,1.507148
14942,,Dorohedoro,38668,TV,"[Action, Comedy, Fantasy, Horror]",8.06,8.048611,2020.0,2020.0,0.923077,0.487813,1.482162


In [ ]:
df = recommender.get_anime_recommendations('Akame ga Kill!', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.505584,1.588881
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.505496,1.568690
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.515085,1.566333
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.505834,1.539376
19600,,Jigokuraku,46569,TV,"[Action, Adventure, Fantasy]",8.26,8.224549,2023.0,2023.0,1.000000,0.501412,1.527906
10527,,Noragami Aragoto,30503,TV,"[Action, Fantasy]",8.16,8.156397,2015.0,2015.0,0.794872,0.551604,1.512833
11043,,Magi: Sinbad no Bouken (TV),31741,TV,"[Action, Adventure, Fantasy]",7.85,7.839802,2016.0,2016.0,0.820513,0.512621,1.453461
8268,,Noragami,20507,TV,"[Action, Fantasy]",7.95,7.947757,2014.0,2014.0,0.769231,0.508175,1.453136
7354,,Toaru Kagaku no Railgun S,16049,TV,"[Action, Fantasy, Sci-Fi]",8.02,8.005639,2013.0,2013.0,0.743590,0.503693,1.451906
11291,,D.Gray-man Hallow,32370,TV,"[Action, Adventure, Fantasy]",7.70,7.671143,2016.0,2016.0,0.820513,0.509217,1.430907


In [ ]:
df = recommender.get_anime_recommendations('Detective Conan', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
20949,,Dr. Stone: New World,48549,TV,"[Adventure, Comedy, Sci-Fi]",8.25,8.205729,2023.0,2023.0,1.000000,0.552148,1.560148
16299,,Dr. Stone: Stone Wars,40852,TV,"[Adventure, Comedy, Sci-Fi]",8.17,8.164181,2021.0,2021.0,0.948718,0.553571,1.545874
14959,,Dr. Stone,38691,TV,"[Adventure, Comedy, Sci-Fi]",8.29,8.286605,2019.0,2019.0,0.897436,0.502220,1.515390
12160,,Detective Conan: Episode One - The Great Detective Turned Small,34036,Special,"[Adventure, Comedy, Mystery]",8.24,8.034132,2016.0,2016.0,0.820513,0.518734,1.480937
5985,,Detective Conan Movie 15: Quarter of Silence,9963,Movie,"[Adventure, Comedy, Mystery]",8.00,7.900168,2011.0,2011.0,0.692308,0.547287,1.458637
21584,,Meitantei Conan: Hannin no Hanzawa-san,50010,TV,"[Comedy, Mystery]",6.80,6.668162,2022.0,2022.0,0.974359,0.668725,1.449785
1368,,Detective Conan Movie 10: Requiem of the Detectives,1506,Movie,"[Adventure, Comedy, Mystery]",8.04,7.951632,2006.0,2006.0,0.564103,0.551315,1.441911
1233,,Detective Conan Movie 05: Countdown to Heaven,1364,Movie,"[Adventure, Comedy, Mystery]",8.12,8.040221,2001.0,2001.0,0.435897,0.554862,1.429312
7148,,Detective Conan Movie 17: Private Eye in the Distant Sea,14735,Movie,"[Adventure, Comedy, Mystery]",7.67,7.569965,2013.0,2013.0,0.743590,0.541679,1.425456
1236,,Detective Conan Movie 08: Magician of the Silver Sky,1367,Movie,"[Adventure, Comedy, Mystery]",8.06,7.972434,2004.0,2004.0,0.512821,0.538254,1.425269


In [ ]:
df = recommender.get_anime_recommendations('Tensei shitara Slime Datta Ken', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
15568,,Tensei shitara Slime Datta Ken 2nd Season,39551,TV,"[Action, Adventure, Comedy, Fantasy]",8.39,8.382957,2021.0,2021.0,0.943396,0.667592,1.648596
16626,,Tensei shitara Slime Datta Ken 2nd Season Part 2,41487,TV,"[Action, Adventure, Comedy, Fantasy]",8.33,8.321181,2021.0,2021.0,0.943396,0.649520,1.628894
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[Action, Adventure, Fantasy]",9.07,9.048079,2022.0,2022.0,0.962264,0.514879,1.628340
19600,,Jigokuraku,46569,TV,"[Action, Adventure, Fantasy]",8.26,8.224549,2023.0,2023.0,0.981132,0.523541,1.539180
16627,,Tensura Nikki: Tensei shitara Slime Datta Ken,41488,TV,"[Comedy, Fantasy]",7.59,7.574482,2021.0,2021.0,0.943396,0.561880,1.479696
11043,,Magi: Sinbad no Bouken (TV),31741,TV,"[Action, Adventure, Fantasy]",7.85,7.839802,2016.0,2016.0,0.849057,0.520858,1.464771
13221,,Lupin III: Part 5,35857,TV,"[Action, Adventure, Comedy, Mystery]",8.13,7.955901,2018.0,2018.0,0.886792,0.483682,1.460971
10865,,Drifters,31339,TV,"[Action, Adventure, Comedy, Fantasy]",7.90,7.889898,2016.0,2016.0,0.849057,0.506194,1.460811
12429,,Nanatsu no Taizai: Imashime no Fukkatsu,34577,TV,"[Action, Adventure, Fantasy]",7.59,7.586863,2018.0,2018.0,0.886792,0.525673,1.445240
11291,,D.Gray-man Hallow,32370,TV,"[Action, Adventure, Fantasy]",7.70,7.671143,2016.0,2016.0,0.849057,0.516394,1.441496


In [ ]:
df = recommender.get_anime_recommendations('Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
20964,,86 Part 2,48569,TV,"[Action, Drama, Sci-Fi]",8.71,8.692836,2021.0,2021.0,0.977778,0.476150,1.562478
8855,,Shigatsu wa Kimi no Uso,23273,TV,"[Drama, Romance]",8.65,8.646914,2014.0,2014.0,0.822222,0.479830,1.528358
16608,,86,41457,TV,"[Action, Drama, Sci-Fi]",8.28,8.270491,2021.0,2021.0,0.977778,0.477433,1.512669
21948,,Lycoris Recoil,50709,TV,[Action],8.20,8.183950,2022.0,2022.0,1.000000,0.472867,1.503623
9604,,Plastic Memories,27775,TV,"[Drama, Romance, Sci-Fi]",7.91,7.904120,2015.0,2015.0,0.844444,0.517298,1.469146
3375,,Kidou Senshi Gundam 00 Second Season,3927,TV,"[Action, Drama, Sci-Fi]",8.08,8.048096,2008.0,2008.0,0.688889,0.480428,1.430240
11072,,Kiznaiver,31798,TV,"[Drama, Romance, Sci-Fi]",7.38,7.374965,2016.0,2016.0,0.866667,0.522201,1.413426
16309,,SSSS.Dynazenon,40870,TV,"[Action, Sci-Fi]",7.42,7.375668,2021.0,2021.0,0.977778,0.479599,1.406763
10540,,Soukyuu no Fafner: Dead Aggressor - Exodus Part 2,30549,TV,"[Action, Drama, Sci-Fi]",7.61,7.309665,2015.0,2015.0,0.844444,0.515613,1.396666
16,,Texhnolyze,26,TV,"[Action, Drama, Sci-Fi]",7.76,7.717234,2003.0,2003.0,0.577778,0.517278,1.393373


In [ ]:
df = recommender.get_anime_recommendations('Kimetsu no Yaiba', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,year,recency_score,similarity_score,final_score
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[Action, Fantasy]",8.49,8.467293,2023.0,2023.0,1.000000,0.675826,1.675637
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[Action, Fantasy]",8.80,8.794506,2021.0,2021.0,0.948718,0.627257,1.671619
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[Action, Fantasy]",8.39,8.380549,2021.0,2021.0,0.948718,0.627586,1.622168
16236,,Jujutsu Kaisen,40748,TV,"[Action, Award Winning, Fantasy]",8.64,8.637287,2020.0,2020.0,0.923077,0.510968,1.568548
16060,,Kimetsu no Yaiba Movie: Mugen Ressha-hen,40456,Movie,"[Action, Fantasy]",8.62,8.615747,2020.0,2020.0,0.923077,0.493573,1.554134
18172,,Chainsaw Man,44511,TV,"[Action, Fantasy]",8.59,8.584003,2022.0,2022.0,0.974359,0.456532,1.535394
6589,,Fate/Zero 2nd Season,11741,TV,"[Action, Fantasy, Supernatural]",8.55,8.544452,2012.0,2012.0,0.717949,0.460878,1.482321
14835,,Toaru Kagaku no Railgun T,38481,TV,"[Action, Fantasy, Sci-Fi]",8.17,8.134640,2020.0,2020.0,0.923077,0.454360,1.469737
9821,,Fate/stay night: Unlimited Blade Works 2nd Season,28701,TV,"[Action, Fantasy, Supernatural]",8.32,8.313739,2015.0,2015.0,0.794872,0.453259,1.464839
10527,,Noragami Aragoto,30503,TV,"[Action, Fantasy]",8.16,8.156397,2015.0,2015.0,0.794872,0.449990,1.443735


While the earlier version of the recommender provided reasonable content-based recommendations, it lacked nuanced control over temporal relevance and user-perceived freshness of anime titles. Specifically, although a recency boost was introduced, it treated all past anime equally without factoring in the anime's current airing status or episode count. This led to scenarios where long-running or still-airing shows were penalized unfairly, and older titles with high episode counts were underrepresented despite still holding popularity. Moreover, there was no mechanism to discourage the system from over-recommending outdated or short-lived anime. To resolve these issues, the updated class introduces a refined age penalty mechanism, dynamically adjusting scores based on whether the anime is currently airing and how many episodes it has. This creates a fairer scoring environment where timeless titles aren't overly suppressed, and still-airing or large-episode anime retain their momentum in the recommendation list. As a result, recommendations now better reflect both content similarity and user-relevant freshness, offering a more practical and satisfying user experience.

#### **AnimeRecommender Class – Advanced Multi-Factor Hybrid Recommendation System**

### **Summary**:

The `AnimeRecommender` class implements an **intelligent, multi-layered recommendation system** for anime titles. It integrates **content-based similarity**, **recency weighting**, **user behavior**, and **temporal penalty modeling** to deliver **personalized and high-quality results**.

---

### **Methodology Overview**:

The system follows a structured **content-based recommendation** approach with **natural language processing**, **multi-hot encoding**, and **temporal scoring**. Its major components include:

---

### **Core Components & Workflow**:

#### 1. **Initialization & Data Assignment**

* Accepts two dataframes:

  * `anime_df`: Metadata about anime titles.
  * `ratings_df`: User rating data.
* Initializes key NLP tools:

  * NLTK-based `WordNetLemmatizer`
  * `stopwords`, `tokenizers`, and `POS taggers`

---

#### 2. **Data Preprocessing (`_preprocess_data`)**

* Cleans and converts:

  * `Score`, `Scored By`, and `Episodes` columns to numeric.
* Extracts:

  * **Release year** from `Aired`.
  * **Multi-label fields**: `Genres`, `Studios`.
* Computes:

  * **IMDb-style weighted rating**:

  $$
  \text{weighted_rating} = \frac{v}{v + m}R + \frac{m}{v + m}C
  $$

---

#### 3. **Feature Engineering (`_create_feature_matrices`)**

Encodes core features into machine-readable format:

| Feature           | Encoding Method                  |
| ----------------- | -------------------------------- |
| Genres, Studios   | `MultiLabelBinarizer`            |
| Type, Source      | `OneHotEncoder`                  |
| Episodes (Binned) | Binned + `OneHotEncoder`         |
| Synopsis          | `TF-IDF` with NLTK preprocessing |

* **Text cleaning uses**:

  * Tokenization
  * Stopword removal
  * POS tagging
  * Lemmatization

---

#### 4. **Similarity Calculation (`_calculate_anime_similarity`)**

Calculates **weighted cosine similarity** between all anime using:

| Feature           | Weight |
| ----------------- | ------ |
| Genres            | 0.30   |
| Synopsis (TF-IDF) | 0.35   |
| Type              | 0.15   |
| Studios           | 0.10   |
| Episodes          | 0.05   |
| Source            | 0.05   |

Returns top N most similar anime based on the composite similarity score.

---

#### 5. **Recency Boost (`_calculate_recency_boost`)**

Scales anime by how recent they are:

$$
\text{recency_boost} = \frac{(\text{year} - \text{min})}{\text{max} - \text{min}}
$$

Used to amplify new titles in final score.

---

#### 6. **Age Penalty (in `get_anime_recommendations1/2`)**

Introduced to penalize:

* Old, short, completed series
* While **rewarding** long or currently airing ones

---

#### 7. **Final Score Combination**

The final score formula blends:

* `similarity_score`
* `weighted_rating`
* `recency_score`
* **Age penalty** (divides or subtracts)
---

### 🔍 **Available Recommendation Modes**:

| Method                          | Description                                                               |
| ------------------------------- | ------------------------------------------------------------------------- |
| `get_anime_recommendations`     | Basic similarity + recency weighting                                      |
| `get_anime_recommendations1`    | Adds age penalty for smarter ranking                                      |
| `get_anime_recommendations2`    | Alternative penalty scoring logic                                         |
| `get_recommendations_by_genres` | Genre-driven recommendations with IMDb-style popularity sorting           |


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')
nltk.download('punkt_tab')

class AnimeRecommender:
    def __init__(self, anime_df, ratings_df):
        """Initialize the recommender with anime and ratings data."""
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        # Initialize NLP tools
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self._preprocess_data()
        self._create_feature_matrices()

        # Similarity weights
        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        """Data cleaning and feature engineering with memory optimization."""
        # Handle missing values
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)

        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')

        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)

        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(lambda x: x.split(', ') if x != 'UNKNOWN' else [])

        # Weighted rating
        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        """Create sparse feature matrices for fast similarity calculations."""
        # Multi-label genres and studios
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        # TF-IDF for synopsis with enhanced cleaning
        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    @staticmethod
    def get_wordnet_pos(tag):
        """Map NLTK POS tag to WordNet POS tag for lemmatization."""
        if tag.startswith('J'):
            return 'a'  # adjective
        elif tag.startswith('V'):
            return 'v'  # verb
        elif tag.startswith('N'):
            return 'n'  # noun
        elif tag.startswith('R'):
            return 'r'  # adverb
        else:
            return 'n'  # default to noun

    def _lemmatize_text(self, text: str) -> str:
        """Lemmatize and clean a single text string."""
        text = text.lower()
        tokens = word_tokenize(text)
        tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
        cleaned_tokens = []
        for token, tag in tagged_tokens:
            if token not in string.punctuation and token not in self.stop_words:
                wordnet_pos = self.get_wordnet_pos(tag)
                lemma = self.lemmatizer.lemmatize(token, pos=wordnet_pos)
                cleaned_tokens.append(lemma)
        return ' '.join(cleaned_tokens)

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        """Clean text data by lowercasing, tokenizing, removing punctuation and stopwords, and lemmatizing."""
        return text_series.apply(self._lemmatize_text)

    def _calculate_anime_similarity(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        """Calculate weighted similarity for a single anime using sparse operations."""
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        # Ensure that the number of recommendations requested doesn't exceed the number of available animes minus 1 (to exclude self)
        num_recs_to_get = min(52, len(self.anime_df) - 1)
        # Sort by similarity score in descending order and get top N+1 (including self)
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        # Exclude the first element (self) and take the next num_recs_to_get
        top_sim = sorted_sim_scores[1:num_recs_to_get + 1]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices


    def _calculate_recency_boost(self, release_year):
        """Calculate a simple recency boost score based on the release year."""
        # Assuming a linear boost, you can adjust this formula
        current_year = 2025  # Adjust if needed
        # Handle potential NaN or non-numeric release_year
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        year_range = max_year - min_year if max_year != min_year else 1

        # Scale the year to a 0-1 range
        scaled_year = (year - min_year) / year_range

        # You can adjust the recency boost formula based on scaled_year
        # Example: a simple linear boost where newer anime get a higher score
        recency_boost = scaled_year # A value between 0 and 1

        return recency_boost


    def get_anime_recommendations(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        """Generate anime recommendations based on a given title."""
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year']
        ].copy()

        # Apply the recency boost directly to the recommended animes
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['similarity_score'] = sim_scores_top


        rating_weight = 1 - similarity_weight

        # Ensure all weight components are non-negative
        rating_weight = max(0, rating_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        qualified_animes = recommended_animes.copy() # Use a consistent variable name

        # Compute final score
        qualified_animes['final_score'] = (
            (1-similarity_weight) * qualified_animes['weighted_rating'] +
            similarity_weight * qualified_animes['similarity_score'])* (1-recency_weight) + (recency_weight * qualified_animes['recency_score'])



        return qualified_animes.sort_values('final_score', ascending=False).head(n)


    def get_recommendations_by_genres(self, genres, n=10):
        """Generate anime recommendations based on specified genres with IMDb-like popularity.

        Args:
            genres (str or List[str]): A single genre or list of genres to base recommendations on.
            n (int, optional): Number of recommendations to return. Defaults to 10.

        Returns:
            DataFrame: Top n anime recommendations with title, genres, final score, and popularity score.
        """
        # Ensure genres is a list
        if isinstance(genres, str):
            genres = [genres]

        # Validate genres
        valid_genres = set(self.mlb_genres.classes_)
        genres = [g for g in genres if g in valid_genres]
        if not genres:
            raise ValueError("No valid genres provided.")

        # Create a genre vector for input genres
        genre_vector = self.mlb_genres.transform([genres]).toarray()[0]

        # Calculate similarity score based on genre overlap
        similarity_scores = (self.genres_encoded.toarray() @ genre_vector).ravel()

        # Create a copy of the anime DataFrame for processing
        recommendations = self.anime_df[['Name', 'Genres', 'Release_year', 'Episodes', 'Status', 'weighted_rating','Image URL']].copy()
        recommendations['similarity_score'] = similarity_scores

        # Filter out anime with no genre overlap
        recommendations = recommendations[recommendations['similarity_score'] > 0]

        # Calculate age penalty
        current_year = 2025  # Current year as of May 21, 2025
        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

        # Apply age penalty: 0 for still airing, reduced penalty for large anime
        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        # Sort by final score and select top n
        recommendations = recommendations.sort_values(by=['weighted_rating', 'age_penalty'],
        ascending=[ False, True]).head(n)

        # Return relevant columns
        return recommendations[['Image URL','Name', 'Genres', 'age_penalty', 'weighted_rating']]

    def get_anime_recommendations1(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        """Generate anime recommendations based on a given title with age penalty."""
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
        ].copy()

        # Calculate similarity and recency scores
        recommended_animes['similarity_score'] = sim_scores_top
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)

        # Calculate age penalty
        current_year = 2025  # Current year as of May 25, 2025
        recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(current_year)
        recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommended_animes['age_penalty'] = recommended_animes.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        # Ensure weights are non-negative
        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        # Compute final score incorporating age penalty
        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
            similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        ) / (1 + recommended_animes['age_penalty'] / 10)  # Scale down age penalty impact

        # Sort by final score and select top n
        return recommended_animes.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'final_score', 'age_penalty']
        ]

    def get_anime_recommendations2(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        """Generate anime recommendations based on a given title with age penalty."""
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
                ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
            ].copy()

            # Calculate similarity and recency scores
        recommended_animes['similarity_score'] = sim_scores_top
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)

            # Calculate age penalty
        current_year = 2025  # Current year as of May 25, 2025
        recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(current_year)
        recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommended_animes['age_penalty'] = recommended_animes.apply(
                lambda row: 0 if row['Status'] == 'Currently Airing' else
                (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
                axis=1
            )

            # Ensure weights are non-negative
        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

            # Compute final score incorporating age penalty
        recommended_animes['final_score'] = (
                (rating_weight * recommended_animes['weighted_rating'] +
                similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) )+(
                recency_weight * recommended_animes['age_penalty']
            )

            # Sort by final score and select top n
        return recommended_animes.sort_values('final_score', ascending=False).head(n)[
                ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'final_score', 'age_penalty']
            ]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


* Lets make an instance and show some test...

In [ ]:
AnimeRecommender = AnimeRecommender(anime_df, ratings_df)
df = AnimeRecommender.get_anime_recommendations1('Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
21948,,Lycoris Recoil,50709,TV,[Action],8.20,8.183950,1.166947,2.837500
20964,,86 Part 2,48569,TV,"[Action, Drama, Sci-Fi]",8.71,8.692836,1.130083,3.800000
16608,,86,41457,TV,"[Action, Drama, Sci-Fi]",8.28,8.270491,1.092670,3.816667
22263,,Engage Kiss,51417,TV,"[Action, Comedy, Romance]",6.84,6.829794,1.048531,2.837500
16309,,SSSS.Dynazenon,40870,TV,"[Action, Sci-Fi]",7.42,7.375668,1.017247,3.800000
21073,,Cardfight!! Vanguard: overDress Season 2,48862,TV,"[Action, Drama]",6.67,6.538178,0.946119,3.783333
16266,,Hypnosis Mic: Division Rap Battle - Rhyme Anima,40803,TV,"[Action, Sci-Fi]",6.80,6.764323,0.902663,4.729167
12271,,Grancrest Senki,34279,TV,"[Action, Drama, Fantasy, Romance]",7.22,7.208601,0.849723,6.300000
11508,,Senki Zesshou Symphogear AXZ,32836,TV,"[Action, Sci-Fi]",7.60,7.472081,0.798264,7.566667
13591,,Beatless,36516,TV,"[Action, Drama, Romance, Sci-Fi]",6.21,6.215648,0.776584,6.416667


In [ ]:
df = AnimeRecommender.get_anime_recommendations2('Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
2013,,Muteki Choujin Zanbot 3,2200,TV,"[Action, Drama, Sci-Fi]",7.16,6.752812,9.855553,43.400000
4292,,Uchuu Kuubo Blue Noah,5763,TV,"[Action, Drama, Sci-Fi]",6.53,6.417003,9.404405,41.400000
59,,Kidou Senshi Gundam,80,TV,"[Action, Drama, Sci-Fi]",7.76,7.718750,8.803062,37.758333
988,,Macross,1088,TV,"[Action, Romance, Sci-Fi]",7.90,7.828247,8.575618,36.550000
3675,,Choujikuu Kidan Southern Cross,4503,TV,"[Action, Drama, Sci-Fi]",6.56,6.497439,8.550448,37.070833
4663,,Chou Kousoku Galvion,6636,TV,"[Action, Sci-Fi]",6.04,6.330598,8.534154,37.241667
2369,,Soukou Kihei Votoms,2582,TV,"[Action, Drama, Sci-Fi]",7.70,7.395743,7.793458,32.900000
64,,Kidou Senshi Zeta Gundam,85,TV,"[Drama, Romance, Sci-Fi]",7.90,7.828992,7.596149,31.666667
1325,,Uchuu no Kishi Tekkaman Blade,1459,TV,"[Action, Adventure, Drama, Romance, Sci-Fi]",7.45,7.190315,6.436900,26.262500
195,,Kidou Senkan Nadesico,218,TV,"[Action, Romance, Sci-Fi]",7.49,7.397964,6.396828,25.858333


In [ ]:
df = AnimeRecommender.get_anime_recommendations('Naruto: Shippuuden', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,Release_year,recency_score,similarity_score,final_score
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[Action, Adventure, Fantasy]",9.07,9.048079,2022.0,0.972222,0.611729,1.696190
10,,Naruto,20,TV,"[Action, Adventure, Fantasy]",7.99,7.988501,2002.0,0.787037,0.783057,1.648506
6456,,Hunter x Hunter (2011),11061,TV,"[Action, Adventure, Fantasy]",9.04,9.037173,2011.0,0.870370,0.555852,1.636514
3961,,Fullmetal Alchemist: Brotherhood,5114,TV,"[Action, Adventure, Drama, Fantasy]",9.10,9.097636,2009.0,0.851852,0.523833,1.618293
368,,Yuu☆Yuu☆Hakusho,392,TV,"[Action, Fantasy]",8.46,8.448490,1992.0,0.694444,0.601456,1.561698
245,,Bleach,269,TV,"[Action, Adventure, Fantasy]",7.92,7.917476,2004.0,0.805556,0.654016,1.555939
11,,One Piece,21,TV,"[Action, Adventure, Fantasy]",8.69,8.686696,1999.0,0.759259,0.519085,1.547233
12428,,Black Clover,34572,TV,"[Action, Comedy, Fantasy]",8.14,8.136070,2017.0,0.925926,0.560497,1.542652
115,,Hunter x Hunter,136,TV,"[Action, Adventure, Fantasy]",8.41,8.397139,1999.0,0.759259,0.554684,1.536694
9225,,Akatsuki no Yona,25013,TV,"[Action, Adventure, Fantasy, Romance]",8.03,8.022766,2014.0,0.898148,0.567099,1.527989


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('High School DxD', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
15566,,Yahari Ore no Seishun Love Comedy wa Machigatteiru. Kan,39547,TV,"[Comedy, Romance]",8.36,8.350713,1.047413,4.750000
14836,,Ore wo Suki nano wa Omae dake ka yo,38483,TV,"[Comedy, Romance]",7.32,7.312971,0.903399,5.700000
16209,,Monster Musume no Oishasan,40708,TV,"[Comedy, Fantasy, Romance, Ecchi]",6.53,6.527310,0.897358,4.750000
12273,,High School DxD Hero,34281,TV,"[Action, Comedy, Romance, Ecchi]",7.25,7.244283,0.896247,6.650000
15419,,Kawaikereba Hentai demo Suki ni Natte Kuremasu ka?,39326,TV,"[Comedy, Romance, Ecchi]",6.50,6.498662,0.858891,5.700000
10586,,Saenai Heroine no Sodatekata ♭,30727,TV,"[Comedy, Romance, Ecchi]",7.76,7.746675,0.846892,7.633333
9152,,High School DxD BorN,24703,TV,"[Action, Comedy, Romance, Ecchi]",7.42,7.416314,0.816511,9.500000
12969,,Imouto sae Ireba Ii.,35413,TV,"[Comedy, Romance, Ecchi]",7.28,7.269535,0.816102,7.600000
12272,,Gamers!,34280,TV,"[Comedy, Romance]",6.76,6.758171,0.769588,7.600000
8856,,Saenai Heroine no Sodatekata,23277,TV,"[Comedy, Romance, Ecchi]",7.48,7.473676,0.750873,9.500000


In [ ]:
df = AnimeRecommender.get_recommendations_by_genres(['Action', 'Fantasy']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,age_penalty,weighted_rating
3961,,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy]",11.733333,9.097636
16617,,Bleach: Sennen Kessen-hen,"[Action, Adventure, Fantasy]",2.837500,9.048079
14865,,Shingeki no Kyojin Season 3 Part 2,"[Action, Drama]",5.750000,9.046816
9880,,Gintama°,"[Action, Comedy, Sci-Fi]",7.875000,9.040355
6456,,Hunter x Hunter (2011),"[Action, Adventure, Fantasy]",7.000000,9.037173
22348,,Shingeki no Kyojin: The Final Season - Kanketsu-hen,"[Action, Drama, Suspense]",0.000000,9.020218
5989,,Gintama',"[Action, Comedy, Sci-Fi]",11.025000,9.019494
7240,,Gintama': Enchousen,"[Action, Comedy, Sci-Fi]",12.295833,9.000788
15525,,Gintama: The Final,"[Action, Comedy, Drama, Sci-Fi]",3.983333,8.968517
12179,,Gintama.,"[Action, Comedy, Sci-Fi]",7.600000,8.947376


* **Install spaCy**


This downloads the **small English language model** (`en_core_web_sm`) which includes:

* Vocabulary
* Syntax
* Named entities

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


The following AnimeRecommender class solves key limitations in the previous version by replacing the outdated NLTK pipeline with a cleaner and more efficient spaCy NLP model, improving genre and studio normalization, and introducing consistent recency scoring and age penalty handling. It also adds personalized user-based recommendations and a simpler genre-based filtering method. These changes lead to smarter, fresher, and more relevant recommendations that better reflect user preferences and anime trends.

---

#### **AnimeRecommender Class — Hybrid NLP & Metadata-Based Anime Recommendation Engine**


### **Overview**:

The `AnimeRecommender` class implements a **hybrid recommendation engine** that blends **NLP-enhanced content filtering**, **metadata-driven similarity**, and **user-based collaborative signals**. It dynamically adjusts recommendation priorities using **recency scoring** and **age-based penalties**, ensuring results are **relevant, personalized, and trend-aware**.


### **Class Methodology & Core Components**:


####  `__init__()`

* Loads `anime_df` and `ratings_df` datasets.
* Initializes spaCy NLP pipeline and vector encoders.
* Applies preprocessing and feature engineering.
* Sets default **feature weights**:

```python
weights = {
    'genres': 0.30,
    'synopsis': 0.35,
    'type': 0.15,
    'studios': 0.10,
    'episodes': 0.05,
    'source': 0.05
}
```


#### `_preprocess_data()`

* Cleans numerical and string fields.
* Extracts `Release_year` from `Aired`.
* Converts genres and studios to normalized lists.
* Calculates **IMDb-style weighted rating**:

$$
\text{weighted\_rating} = \left( \frac{v}{v + m} \right) R + \left( \frac{m}{v + m} \right) C
$$

Where:

* $R$: anime's mean score
* $v$: number of votes (Scored By)
* $m$: 65th percentile of all vote counts
* $C$: global mean score

---

#### `_create_feature_matrices()`

Creates efficient encodings for similarity computation:

| Feature    | Encoding Method              |
| ---------- | ---------------------------- |
| `Genres`   | MultiLabelBinarizer (sparse) |
| `Studios`  | MultiLabelBinarizer          |
| `Type`     | OneHotEncoder                |
| `Source`   | OneHotEncoder                |
| `Episodes` | Binned + OneHotEncoder       |
| `Synopsis` | spaCy → Lemmatized → TF-IDF  |

---

### 📊 **Similarity Calculation**

#### `_calculate_anime_similarity(idx)`

Generates **weighted similarity** score:

$$
\text{similarity} = \sum w_i \cdot \text{sim}_i
$$

Where $w_i$ are feature weights and $\text{sim}_i$ is cosine or binary match similarity for:

* Genres
* Synopsis
* Type
* Studios
* Episodes
* Source

---

### **Recency Scoring**

#### `_calculate_recency_boost(release_year)`

Applies **min-max normalization** on release year:

$$
\text{recency_score} = \frac{\text{year} - \text{min\_year}}{\text{max_year} - \text{min_year}}
$$

---

### **Final Scoring Formulas**

#### 1. Without Age Penalty (in `get_anime_recommendations()`):

$$\text{final_score} = \left[ (1 - w_s) \cdot \text{weighted_rating} + w_s \cdot \text{similarity} \right] \cdot (1 - w_r) + w_r \cdot \text{recency_score}
$$

#### 2. With Age Penalty (in `get_anime_recommendations1()` and `2()`):

$$
\text{final_score} = \frac{
\left[ (1 - w_s) \cdot \text{weighted_rating} + w_s \cdot \text{similarity} \right] \cdot (1 - w_r) + w_r \cdot \text{recency_score}
}{1 + \frac{\text{age_penalty}}{10}}
$$

---

###  **Recommendation Strategies**

| Method                          | Description                                                               |
| ------------------------------- | ------------------------------------------------------------------------- |
| `get_anime_recommendations`     | Based on title; includes similarity, rating, and recency boost.           |
| `get_anime_recommendations1/2`  | Adds **age\_penalty** to suppress outdated content unless still airing.   |
| `get_user_recommendations`      | Based on user’s top-rated anime; scores by similarity × rating × recency. |
| `get_recommendations_by_genres` | Genre-based matching using binary or semantic overlap.                    |
| `simple()`                      | Lightweight genre intersection without encoders.                          |

---

###  **Design Highlights**

| Feature                                               | Included |
| ----------------------------------------------------- | -------- |
|  Weighted metadata-based similarity (multi-feature) | ✅        |
|  NLP-powered synopsis matching via spaCy + TF-IDF   | ✅        |
|  Recency boosting to promote newer anime            | ✅        |
|  Age penalty to deprioritize outdated shows        | ✅        |
|  IMDb-like popularity-adjusted scoring               | ✅        |
|  Modular functions for extensibility                | ✅        |

In [ ]:
class AnimeRecommender:
    def __init__(self, anime_df, ratings_df):
        self.nlp = spacy.load("en_core_web_sm")
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self.nlp = spacy.load("en_core_web_sm")
        self.current_year = datetime.now().year

        self._preprocess_data()
        self._create_feature_matrices()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)
        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')
        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)
        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        def normalize_genre_list(genre_string):
            if genre_string == 'UNKNOWN' or pd.isna(genre_string):
                return []
            genres = re.split(r',\s*', genre_string)
            cleaned = [re.sub(r'[^\w\s]', '', g).strip().lower() for g in genres]
            return list(set(cleaned))

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(normalize_genre_list)
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(normalize_genre_list)

        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        return text_series.apply(self._spacy_clean)

    def _spacy_clean(self, text: str) -> str:
        doc = self.nlp(text.lower())
        return ' '.join(
            token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
        )

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        year_range = max_year - min_year if max_year != min_year else 1
        scaled_year = (year - min_year) / year_range
        return scaled_year

    def _calculate_anime_similarity(self, idx: int):
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        num_recs_to_get = min(52, len(self.anime_df) - 1)
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:num_recs_to_get + 1]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_user_recommendations(self, user_id, n=10, top_k=20):
        user_ratings = self.ratings_df[self.ratings_df['user_id'] == user_id]
        if user_ratings.empty:
            return pd.DataFrame()

        top_rated = (
            user_ratings
            .merge(self.anime_df, on='anime_id')
            .nlargest(top_k, 'rating')
            [['anime_id', 'rating', 'Release_year']]
        )

        candidate_scores = defaultdict(float)
        anime_id_to_index = pd.Series(self.anime_df.index, index=self.anime_df['anime_id']).to_dict()

        for _, row in top_rated.iterrows():
            anime_id = row['anime_id']
            if anime_id in anime_id_to_index:
                anime_idx = anime_id_to_index[anime_id]
                sim_scores_top, top_indices = self._calculate_anime_similarity(anime_idx)
                recency = self._calculate_recency_boost(row['Release_year'])
                weighted_scores = sim_scores_top * row['rating'] * recency

                for i, rec_idx in enumerate(top_indices):
                    if rec_idx < len(self.anime_df):
                        candidate_scores[rec_idx] += weighted_scores[i]

        watched_ids = set(user_ratings['anime_id'])
        recommended_anime_indices = sorted(candidate_scores.keys(), key=lambda k: candidate_scores[k], reverse=True)
        recommended_anime_indices = [idx for idx in recommended_anime_indices if self.anime_df.iloc[idx]['anime_id'] not in watched_ids]
        top_n_indices = recommended_anime_indices[:n]

        recommendations = (
            self.anime_df
            .iloc[top_n_indices]
            .assign(predicted_score=lambda x: x.index.map(candidate_scores))
            [['Name', 'Image URL', 'anime_id', 'Type', 'Genres', 'Score', 'predicted_score']]
        )

        return recommendations

    def get_anime_recommendations(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year']
        ].copy()

        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['similarity_score'] = sim_scores_top

        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
             similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        )

        return recommended_animes.sort_values('final_score', ascending=False).head(n)

    def get_anime_recommendations1(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
        ].copy()

        recommended_animes['similarity_score'] = sim_scores_top
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(self.current_year)
        recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommended_animes['age_penalty'] = recommended_animes.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
             similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        ) / (1 + recommended_animes['age_penalty'] / 10)

        return recommended_animes.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'final_score', 'age_penalty']
        ]

    def get_anime_recommendations2(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
        ].copy()

        recommended_animes['similarity_score'] = sim_scores_top
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(self.current_year)
        recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommended_animes['age_penalty'] = recommended_animes.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
             similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        ) / (1 + recommended_animes['age_penalty'] / 10)

        return recommended_animes.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'final_score', 'age_penalty']
        ]

    def get_recommendations_by_genres(self, genres, n=10):
        if isinstance(genres, str):
            genres = [genres]

        valid_genres = set(self.mlb_genres.classes_)
        genres = [g.strip().lower() for g in genres if g.strip().lower() in valid_genres]
        if not genres:
            raise ValueError("No valid genres provided.")

        genre_vector = self.mlb_genres.transform([genres]).toarray()[0]
        similarity_scores = (self.genres_encoded.toarray() @ genre_vector).ravel()

        recommendations = self.anime_df.copy()
        recommendations['similarity_score'] = similarity_scores
        recommendations = recommendations[recommendations['similarity_score'] > 0]
        current_year = self.current_year

        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)
        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        recommendations = recommendations.sort_values(
            by=['weighted_rating', 'age_penalty'], ascending=[False, True]
        ).head(n)

        return recommendations[['Image URL', 'Name', 'Genres', 'age_penalty', 'weighted_rating']]

    def get_recommendations_by_genres_simple(self, genres, n=10):
        if isinstance(genres, str):
            genres = [genres]
        input_genres = {g.strip().lower() for g in genres}
        if not input_genres:
            raise ValueError("No valid genres provided.")

        if not isinstance(self.anime_df['Genres'].iloc[0], set):
            self.anime_df['Genres'] = self.anime_df['Genres'].apply(
                lambda g_list: {g.strip().lower() for g in g_list}
            )

        self.anime_df['genre_similarity'] = self.anime_df['Genres'].apply(
            lambda anime_genres: len(input_genres & anime_genres)
        )

        recommendations = self.anime_df[self.anime_df['genre_similarity'] > 0].copy()
        current_year = self.current_year
        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        recommendations = recommendations.sort_values(
            by=['genre_similarity', 'weighted_rating', 'age_penalty'], ascending=[False, False, True]
        ).head(n)

        return recommendations[['Image URL', 'Name', 'Genres', 'genre_similarity', 'weighted_rating', 'age_penalty']]


In [ ]:
AnimeRecommender = AnimeRecommender(anime_df, ratings_df)

In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Darling in the FranXX', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
21948,,Lycoris Recoil,50709,TV,[action],8.20,8.183950,1.166601,2.837500
20964,,86 Part 2,48569,TV,"[scifi, drama, action]",8.71,8.692836,1.129940,3.800000
16608,,86,41457,TV,"[scifi, drama, action]",8.28,8.270491,1.092477,3.816667
22263,,Engage Kiss,51417,TV,"[action, comedy, romance]",6.84,6.829794,1.048399,2.837500
16309,,SSSS.Dynazenon,40870,TV,"[scifi, action]",7.42,7.375668,1.017294,3.800000
21073,,Cardfight!! Vanguard: overDress Season 2,48862,TV,"[drama, action]",6.67,6.538178,0.945743,3.783333
16266,,Hypnosis Mic: Division Rap Battle - Rhyme Anima,40803,TV,"[scifi, action]",6.80,6.764323,0.902413,4.729167
12271,,Grancrest Senki,34279,TV,"[drama, action, romance, fantasy]",7.22,7.208601,0.849582,6.300000
13591,,Beatless,36516,TV,"[scifi, drama, action, romance]",6.21,6.215648,0.776463,6.416667
8855,,Shigatsu wa Kimi no Uso,23273,TV,"[drama, romance]",8.65,8.646914,0.771948,9.991667


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Darling in the FranXX', n=10)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

TypeError: AnimeRecommender.get_anime_recommendations1() missing 1 required positional argument: 'title'

In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Fate/stay night: Unlimited Blade Works', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
23475,,Dead Mount Death Play,53613,TV,"[action, fantasy, supernatural]",7.31,7.240476,1.407744,0.000000
20893,,"Maou Gakuin no Futekigousha: Shijou Saikyou no Maou no Shiso, Tensei shite Shison-tachi no Gakkou e Kayou II",48417,TV,"[action, fantasy]",6.91,6.884430,1.332888,0.000000
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[action, fantasy]",8.49,8.467293,1.329885,1.908333
21139,,Mahou Shoujo Magical Destroyers,48981,TV,"[action, fantasy]",6.49,6.470741,1.285783,0.000000
24370,,Sinbi Apateu: Zero,55047,TV,"[fantasy, supernatural]",6.39,6.387130,1.272720,0.000000
21905,,Yu☆Gi☆Oh! Go Rush!!,50607,TV,"[action, fantasy]",5.70,6.168963,1.244499,0.000000
18172,,Chainsaw Man,44511,TV,"[action, fantasy]",8.59,8.584003,1.193523,2.850000
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[action, fantasy]",8.80,8.794506,1.171410,3.816667
21790,,Mononogatari,50384,TV,"[action, supernatural]",7.20,7.113428,1.144930,1.900000
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,"[action, fantasy]",6.39,6.387130,1.137021,0.000000


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Kimetsu no Yaiba', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[action, fantasy]",8.49,8.467293,1.405850,1.908333
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"[action, fantasy]",8.80,8.794506,1.213726,3.816667
18172,,Chainsaw Man,44511,TV,"[action, fantasy]",8.59,8.584003,1.194630,2.850000
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[action, fantasy]",8.39,8.380549,1.171226,3.883333
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,"[action, fantasy]",6.39,6.387130,1.137021,0.000000
16236,,Jujutsu Kaisen,40748,TV,"[action, award winning, fantasy]",8.64,8.637287,1.085981,4.500000
22190,,Ragna Crimson,51297,TV,"[action, fantasy]",6.39,6.387130,1.058089,1.983333
21741,,Delicious Party♡Precure,50281,TV,"[action, fantasy]",6.96,6.708747,1.049547,2.437500
16060,,Kimetsu no Yaiba Movie: Mugen Ressha-hen,40456,Movie,"[action, fantasy]",8.62,8.615747,1.042566,4.979167
17997,,Tropical-Rouge! Precure,44191,TV,"[action, fantasy]",7.41,7.052018,1.021119,3.233333


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('One Piece', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
23543,,Hirogaru Sky! Precure,53716,TV,"[action, fantasy]",7.69,6.923455,1.408175,0.000000
21578,,Edens Zero 2nd Season,50002,TV,"[scifi, action, adventure, fantasy]",7.43,7.164798,1.407991,0.000000
24186,,Mahoutsukai Precure! 2,54717,TV,"[action, fantasy]",6.39,6.387130,1.214710,0.991667
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[action, adventure, fantasy]",7.74,7.643075,1.182576,2.916667
21472,,Fairy Tail: 100 Years Quest,49785,TV,"[action, adventure, fantasy]",6.39,6.387130,1.143662,0.000000
24174,,Boruto: Naruto Next Generations Part 2,54687,TV,"[action, adventure, fantasy]",6.39,6.387130,1.140456,0.000000
22481,,Nanatsu no Taizai: Mokushiroku no Yonkishi,51794,TV,"[action, adventure, fantasy]",6.39,6.387130,1.121166,1.983333
24042,,Ishura,54449,TV,"[action, adventure, fantasy]",6.39,6.387130,1.121126,0.000000
24175,,Naruto (Shinsaku Anime),54688,TV,"[action, adventure, fantasy]",6.39,6.387130,1.117063,1.966667
22788,,"Shangri-La Frontier: Kusoge Hunter, Kamige ni Idoman to su",52347,TV,"[action, adventure, fantasy]",6.39,6.387130,1.116004,1.983333


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Bleach: Sennen Kessen-hen', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
19600,,Jigokuraku,46569,TV,"[action, adventure, fantasy]",8.26,8.224549,1.563704,0.000000
11,,One Piece,21,TV,"[action, adventure, fantasy]",8.69,8.686696,1.539619,0.000000
22017,,Saikyou Onmyouji no Isekai Tenseiki,50932,TV,"[action, adventure, fantasy]",7.20,7.181995,1.179488,1.891667
23754,,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,53998,TV,"[action, adventure, fantasy]",6.39,6.387130,1.143882,1.983333
21472,,Fairy Tail: 100 Years Quest,49785,TV,"[action, adventure, fantasy]",6.39,6.387130,1.112952,0.000000
22481,,Nanatsu no Taizai: Mokushiroku no Yonkishi,51794,TV,"[action, adventure, fantasy]",6.39,6.387130,1.090871,1.983333
12428,,Black Clover,34572,TV,"[action, comedy, fantasy]",8.14,8.136070,1.076231,4.000000
22235,,Orient: Awajishima Gekitou-hen,51368,TV,"[action, adventure, fantasy]",7.04,6.978657,1.072318,2.850000
16630,,Nanatsu no Taizai: Funnu no Shinpan,41491,TV,"[action, adventure, fantasy]",6.58,6.578161,1.002656,3.600000
12427,,Boruto: Naruto Next Generations,34566,TV,"[action, adventure, fantasy]",6.06,6.061365,0.947646,4.000000


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Hunter x Hunter (2011)', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
11,,One Piece,21,TV,"[action, adventure, fantasy]",8.69,8.686696,1.539138,0.000000
21093,,Overlord IV,48895,TV,"[action, adventure, fantasy]",8.09,8.076119,1.198494,2.837500
22758,,Ore dake Level Up na Ken,52299,TV,"[action, adventure, fantasy]",6.39,6.387130,1.191686,0.991667
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[action, adventure, fantasy]",7.74,7.643075,1.148750,2.916667
21472,,Fairy Tail: 100 Years Quest,49785,TV,"[action, adventure, fantasy]",6.39,6.387130,1.116855,0.000000
10577,,Dragon Ball Super,30694,TV,"[action, comedy, adventure, fantasy]",7.43,7.426386,0.946701,5.000000
13260,,Fairy Tail: Final Series,35972,TV,"[action, adventure, fantasy]",7.57,7.561180,0.946578,5.512500
12427,,Boruto: Naruto Next Generations,34566,TV,"[action, adventure, fantasy]",6.06,6.061365,0.922746,4.000000
14331,,Overlord III,37675,TV,"[action, adventure, fantasy]",7.92,7.914924,0.910894,6.620833
8558,,Fairy Tail (2014),22043,TV,"[action, adventure, fantasy]",7.65,7.645865,0.902034,6.325000


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Bleach', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
19600,,Jigokuraku,46569,TV,"[action, adventure, fantasy]",8.26,8.224549,1.533767,0.000000
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[action, adventure, fantasy]",9.07,9.048079,1.403018,2.837500
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[action, adventure, fantasy]",7.74,7.643075,1.150671,2.916667
23754,,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,53998,TV,"[action, adventure, fantasy]",6.39,6.387130,1.143882,1.983333
12428,,Black Clover,34572,TV,"[action, comedy, fantasy]",8.14,8.136070,1.101288,4.000000
22235,,Orient: Awajishima Gekitou-hen,51368,TV,"[action, adventure, fantasy]",7.04,6.978657,1.071986,2.850000
16630,,Nanatsu no Taizai: Funnu no Shinpan,41491,TV,"[action, adventure, fantasy]",6.58,6.578161,0.976640,3.600000
12427,,Boruto: Naruto Next Generations,34566,TV,"[action, adventure, fantasy]",6.06,6.061365,0.972418,4.000000
6456,,Hunter x Hunter (2011),11061,TV,"[action, adventure, fantasy]",9.04,9.037173,0.962227,7.000000
13260,,Fairy Tail: Final Series,35972,TV,"[action, adventure, fantasy]",7.57,7.561180,0.947759,5.512500


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Kimetsu no Yaiba: Yuukaku-hen', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"[action, fantasy]",8.49,8.467293,1.437633,1.908333
20893,,"Maou Gakuin no Futekigousha: Shijou Saikyou no Maou no Shiso, Tensei shite Shison-tachi no Gakkou e Kayou II",48417,TV,"[action, fantasy]",6.91,6.884430,1.383529,0.000000
23543,,Hirogaru Sky! Precure,53716,TV,"[action, fantasy]",7.69,6.923455,1.374168,0.000000
21905,,Yu☆Gi☆Oh! Go Rush!!,50607,TV,"[action, fantasy]",5.70,6.168963,1.277073,0.000000
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,"[action, fantasy]",8.39,8.380549,1.245945,3.883333
18172,,Chainsaw Man,44511,TV,"[action, fantasy]",8.59,8.584003,1.245837,2.850000
21793,,Mato Seihei no Slave,50392,TV,"[action, ecchi, fantasy]",6.39,6.387130,1.208660,0.991667
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,"[action, fantasy]",6.39,6.387130,1.208456,0.000000
21207,,High Card,49154,TV,"[action, fantasy]",7.14,7.089661,1.169196,1.900000
23669,,Ao no Exorcist (Shin Series),53889,TV,"[action, fantasy]",6.39,6.387130,1.140456,0.000000


In [ ]:
df = AnimeRecommender.get_recommendations_by_genres_simple(['horror', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,genre_similarity,weighted_rating,age_penalty
23,,Kenpuu Denki Berserk,"{horror, drama, action, adventure, fantasy}",2,8.548288,25.083333
708,,Hellsing Ultimate,"{horror, action, supernatural}",2,8.342544,18.208333
8677,,Kiseijuu: Sei no Kakuritsu,"{scifi, horror, action}",2,8.336886,9.900000
6671,,Berserk: Ougon Jidai-hen III - Kourin,"{horror, drama, action, adventure, fantasy}",2,8.176690,11.950000
28,,Akira,"{horror, supernatural, action, scifi, adventure}",2,8.153587,36.845833
14942,,Dorohedoro,"{horror, action, comedy, fantasy}",2,8.048611,4.750000
509,,Vampire Hunter D (2000),"{horror, romance, drama, action, scifi, fantasy}",2,7.880489,24.895833
6670,,Berserk: Ougon Jidai-hen II - Doldrey Kouryaku,"{horror, drama, action, adventure, fantasy}",2,7.850457,12.945833
8619,,Tokyo Ghoul,"{horror, action, fantasy}",2,7.788620,10.450000
12774,,Devilman: Crybaby,"{horror, action, supernatural, avant garde}",2,7.756334,6.708333


In [ ]:
df = AnimeRecommender.get_anime_recommendations1('Chainsaw Man', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
19600,,Jigokuraku,46569,TV,"{action, adventure, fantasy}",8.26,8.224549,1.560568,0.000000
22709,,Mashle,52211,TV,"{action, comedy, fantasy}",7.59,7.553219,1.449163,0.000000
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,"{action, fantasy}",8.49,8.467293,1.336205,1.908333
21139,,Mahou Shoujo Magical Destroyers,48981,TV,"{action, fantasy}",6.49,6.470741,1.317749,0.000000
20972,,Shingeki no Kyojin: The Final Season Part 2,48583,TV,"{drama, action}",8.77,8.763341,1.238309,2.850000
21403,,Chiyu Mahou no Machigatta Tsukaikata: Senjou wo Kakeru Kaifuku Youin,49613,TV,"{action, fantasy}",6.39,6.387130,1.191462,0.991667
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,"{action, fantasy}",8.80,8.794506,1.175615,3.816667
22042,,Jujutsu Kaisen 2nd Season,51009,TV,"{action, fantasy}",6.39,6.387130,1.172255,1.983333
21207,,High Card,49154,TV,"{action, fantasy}",7.14,7.089661,1.170209,1.900000
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,"{action, fantasy}",6.39,6.387130,1.140456,0.000000


In [ ]:
df = AnimeRecommender.get_anime_recommendations2('Tensei shitara Slime Datta Ken', n=24)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
19600,,Jigokuraku,46569,TV,"[fantasy, action, adventure]",8.26,8.224549,1.538809,0.000000
16617,,Bleach: Sennen Kessen-hen,41467,TV,"[fantasy, action, adventure]",9.07,9.048079,1.269965,2.837500
15568,,Tensei shitara Slime Datta Ken 2nd Season,39551,TV,"[fantasy, action, adventure, comedy]",8.39,8.382957,1.197764,3.800000
23458,,Tensei shitara Slime Datta Ken 3rd Season,53580,TV,"[fantasy, action, adventure, comedy]",6.39,6.387130,1.196618,0.991667
16626,,Tensei shitara Slime Datta Ken 2nd Season Part 2,41487,TV,"[fantasy, action, adventure, comedy]",8.33,8.321181,1.184758,3.800000
21510,,Tensei shitara Slime Datta Ken Movie: Guren no Kizuna-hen,49877,Movie,"[fantasy, action, adventure, comedy]",7.63,7.587295,1.107159,2.987500
24677,,Naruto (2023),55453,TV,"[fantasy, action, adventure, comedy]",6.39,6.387130,1.092246,1.983333
21472,,Fairy Tail: 100 Years Quest,49785,TV,"[fantasy, action, adventure]",6.39,6.387130,1.092210,0.000000
24116,,Tensei shitara Slime Datta Ken: Coleus no Yume,54565,OVA,"[fantasy, action, adventure, comedy]",6.39,6.387130,1.077132,1.975000
16627,,Tensura Nikki: Tensei shitara Slime Datta Ken,41488,TV,"[fantasy, comedy]",7.59,7.574482,1.075618,3.800000


In [ ]:
df['Name']

,Name
19600,Jigokuraku
16617,Bleach: Sennen Kessen-hen
15568,Tensei shitara Slime Datta Ken 2nd Season
23458,Tensei shitara Slime Datta Ken 3rd Season
16626,Tensei shitara Slime Datta Ken 2nd Season Part 2
21510,Tensei shitara Slime Datta Ken Movie: Guren no...
24677,Naruto (2023)
21472,Fairy Tail: 100 Years Quest
24116,Tensei shitara Slime Datta Ken: Coleus no Yume
16627,Tensura Nikki: Tensei shitara Slime Datta Ken


The AnimeRecommender1 class builds upon the earlier AnimeRecommender implementation by refining the genre-based recommendation logic and improving the robustness of set operations. Previously, genre filtering was prone to inconsistencies due to format mismatches or missing preprocessing. This updated version resolves that by dynamically converting genre lists into standardized sets, ensuring accurate genre matching regardless of input format. Additionally, the revised logic supports full subset matching using issubset() to only recommend anime that include all user-specified genres. This enhancement makes genre filtering more precise and eliminates false matches—effectively solving a subtle reliability issue in the earlier system.

### **Genre-Based Recommendation Methodology (Improved Genre Subset Matching)**

The enhanced genre-based recommendation method implemented in the `AnimeRecommender1` class adopts a rigorous set-theoretic approach to ensure precise alignment between user-specified genre preferences and the genre composition of candidate anime titles. Unlike prior implementations that relied on partial overlaps or genre count similarity, the improved method enforces a **subset constraint**, thereby ensuring that **only anime whose genre sets fully contain all requested genres** are considered for recommendation.

#### **1. Genre Input Normalization**

User-provided genre queries are first preprocessed into standardized, lowercase string sets:

```python
input_genres = {g.strip().lower() for g in genres}
```

This step mitigates inconsistencies arising from formatting variations and capitalization in the genre metadata.

#### **2. Genre Representation Alignment**

Each anime's `Genres` list is dynamically converted into a Python `set` to enable direct subset comparisons:

```python
genre_sets = self.anime_df['Genres'].apply(lambda g_list: {g.strip().lower() for g in g_list})
```

This ensures that genre comparisons are type-consistent, precise, and scalable across large datasets.

#### **3. Subset Filtering**

A boolean mask is computed for each anime entry, evaluating whether the user-specified genres are a subset of the anime's genres:

```python
self.anime_df['has_all_genres'] = genre_sets.apply(lambda anime_genres: input_genres.issubset(anime_genres))
```

Only anime entries where this condition holds true are retained for further ranking.

#### **4. Temporal Adjustment via Age Penalty**

To discourage overly outdated or long-running anime, an **age penalty** function is applied:

$$
\text{age_penalty} =
\begin{cases}
0 & \text{if status is 'Currently Airing'} \\
(\text{current_year} - \text{end_year}) \cdot \left(1 - 0.5 \cdot \min\left(\frac{\text{episodes}}{120}, 1\right)\right) & \text{otherwise}
\end{cases}
$$

This formula penalizes older anime while accounting for their length, with shorter series receiving reduced penalties.

#### **5. Ranking and Recommendation Output**

The final list is sorted based on:

1. **Weighted rating** (IMDb-style formula integrating rating value and popularity).
2. **Age penalty**, in ascending order.

Only the top-N anime entries that meet the genre subset condition and rank highest after age adjustment are returned as recommendations.

---

### **Comparative Advantage**

This revised methodology improves upon earlier implementations by enforcing stricter semantic coherence between user intent and system output. It avoids the inclusion of marginally related anime and provides a more **targeted and trustworthy recommendation experience**, particularly for users with specific genre preferences.


In [ ]:
class AnimeRecommender1:
    def __init__(self, anime_df, ratings_df):
        self.nlp = spacy.load("en_core_web_sm")
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self.nlp = spacy.load("en_core_web_sm")
        self.current_year = datetime.now().year

        self._preprocess_data()
        self._create_feature_matrices()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)
        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')
        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)
        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        def normalize_genre_list(genre_string):
            if genre_string == 'UNKNOWN' or pd.isna(genre_string):
                return []
            genres = re.split(r',\s*', genre_string)
            cleaned = [re.sub(r'[^\w\s]', '', g).strip().lower() for g in genres]
            return list(set(cleaned))

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(normalize_genre_list)
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(normalize_genre_list)

        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        return text_series.apply(self._spacy_clean)

    def _spacy_clean(self, text: str) -> str:
        doc = self.nlp(text.lower())
        return ' '.join(
            token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
        )

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        year_range = max_year - min_year if max_year != min_year else 1
        scaled_year = (year - min_year) / year_range
        return scaled_year

    def _calculate_anime_similarity(self, idx: int):
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        num_recs_to_get = min(52, len(self.anime_df) - 1)
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:num_recs_to_get + 1]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_user_recommendations(self, user_id, n=10, top_k=20):
        user_ratings = self.ratings_df[self.ratings_df['user_id'] == user_id]
        if user_ratings.empty:
            return pd.DataFrame()

        top_rated = (
            user_ratings
            .merge(self.anime_df, on='anime_id')
            .nlargest(top_k, 'rating')
            [['anime_id', 'rating', 'Release_year']]
        )

        candidate_scores = defaultdict(float)
        anime_id_to_index = pd.Series(self.anime_df.index, index=self.anime_df['anime_id']).to_dict()

        for _, row in top_rated.iterrows():
            anime_id = row['anime_id']
            if anime_id in anime_id_to_index:
                anime_idx = anime_id_to_index[anime_id]
                sim_scores_top, top_indices = self._calculate_anime_similarity(anime_idx)
                recency = self._calculate_recency_boost(row['Release_year'])
                weighted_scores = sim_scores_top * row['rating'] * recency

                for i, rec_idx in enumerate(top_indices):
                    if rec_idx < len(self.anime_df):
                        candidate_scores[rec_idx] += weighted_scores[i]

        watched_ids = set(user_ratings['anime_id'])
        recommended_anime_indices = sorted(candidate_scores.keys(), key=lambda k: candidate_scores[k], reverse=True)
        recommended_anime_indices = [idx for idx in recommended_anime_indices if self.anime_df.iloc[idx]['anime_id'] not in watched_ids]
        top_n_indices = recommended_anime_indices[:n]

        recommendations = (
            self.anime_df
            .iloc[top_n_indices]
            .assign(predicted_score=lambda x: x.index.map(candidate_scores))
            [['Name', 'Image URL', 'anime_id', 'Type', 'Genres', 'Score', 'predicted_score']]
        )

        return recommendations

    def get_anime_recommendations(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year']
        ].copy()

        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['similarity_score'] = sim_scores_top

        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
             similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        )

        return recommended_animes.sort_values('final_score', ascending=False).head(n)

    def get_anime_recommendations1(self, title, n=10, similarity_weight=0.85, recency_weight=0.2):
        matching_animes = self.anime_df[self.anime_df['Name'] == title]
        if matching_animes.empty:
            raise ValueError(f"Anime '{title}' not found in the database.")
        idx = matching_animes.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

        recommended_animes = self.anime_df.iloc[top_indices][
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
        ].copy()

        recommended_animes['similarity_score'] = sim_scores_top
        recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
        recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(self.current_year)
        recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommended_animes['age_penalty'] = recommended_animes.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        rating_weight = max(0, 1 - similarity_weight)
        similarity_weight = max(0, similarity_weight)
        recency_weight = max(0, recency_weight)

        recommended_animes['final_score'] = (
            (rating_weight * recommended_animes['weighted_rating'] +
             similarity_weight * recommended_animes['similarity_score']) * (1 - recency_weight) +
            recency_weight * recommended_animes['recency_score']
        ) / (1 + recommended_animes['age_penalty'] / 10)

        return recommended_animes.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'final_score', 'age_penalty']
        ]

    def get_recommendations_by_genres_simple(self, genres, n=10):
        if isinstance(genres, str):
            genres = [genres]
        input_genres = {g.strip().lower() for g in genres}
        if not input_genres:
            raise ValueError("No valid genres provided.")

        # Always safely convert genre lists to sets on-the-fly
        genre_sets = self.anime_df['Genres'].apply(lambda g_list: {g.strip().lower() for g in g_list})
        self.anime_df['has_all_genres'] = genre_sets.apply(
            lambda anime_genres: input_genres.issubset(anime_genres)
        )

        recommendations = self.anime_df[self.anime_df['has_all_genres']].copy()
        current_year = self.current_year
        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        recommendations = recommendations.sort_values(
            by=['weighted_rating', 'age_penalty'], ascending=[False, True]
        ).head(n)

        return recommendations[['Image URL', 'Name', 'Genres', 'weighted_rating', 'age_penalty']]


##### Lets make an instance and do some tests...

In [ ]:
AnimeRecommender1 = AnimeRecommender1(anime_df, ratings_df)

In [ ]:
df = AnimeRecommender1.get_recommendations_by_genres_simple(['horror', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty
23,,Kenpuu Denki Berserk,"[horror, drama, action, adventure, fantasy]",8.548288,25.083333
708,,Hellsing Ultimate,"[horror, action, supernatural]",8.342544,18.208333
8677,,Kiseijuu: Sei no Kakuritsu,"[scifi, horror, action]",8.336886,9.900000
6671,,Berserk: Ougon Jidai-hen III - Kourin,"[horror, drama, action, adventure, fantasy]",8.176690,11.950000
28,,Akira,"[horror, supernatural, action, scifi, adventure]",8.153587,36.845833
14942,,Dorohedoro,"[horror, action, comedy, fantasy]",8.048611,4.750000
509,,Vampire Hunter D (2000),"[horror, drama, action, scifi, fantasy, romance]",7.880489,24.895833
6670,,Berserk: Ougon Jidai-hen II - Doldrey Kouryaku,"[horror, drama, action, adventure, fantasy]",7.850457,12.945833
8619,,Tokyo Ghoul,"[horror, action, fantasy]",7.788620,10.450000
12774,,Devilman: Crybaby,"[horror, action, supernatural, avant garde]",7.756334,6.708333


In [ ]:
df = AnimeRecommender1.get_recommendations_by_genres_simple([ 'scifi','action','romance',]	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty
509,,Vampire Hunter D (2000),"[horror, drama, action, scifi, fantasy, romance]",7.880489,24.895833
3131,,Macross F,"[scifi, action, award winning, romance]",7.850851,15.229167
988,,Macross,"[scifi, action, romance]",7.828247,36.550000
989,,Macross: Do You Remember Love?,"[scifi, action, romance]",7.823890,40.829167
4922,,Macross F Movie 2: Sayonara no Tsubasa,"[scifi, action, award winning, romance]",7.758669,13.941667
16612,,Date A Live IV,"[scifi, action, romance]",7.746146,2.850000
72,,Kidou Senshi Gundam SEED,"[award winning, drama, action, scifi, romance]",7.724560,18.208333
4074,,Macross F Movie 1: Itsuwari no Utahime,"[scifi, action, romance]",7.646773,15.933333
373,,Seikai no Senki II,"[scifi, action, romance]",7.612668,23.000000
1181,,Urusei Yatsura,"[comedy, drama, action, scifi, adventure, romance]",7.608737,22.000000


In [ ]:
df = AnimeRecommender1.get_anime_recommendations1('Kizumonogatari III: Reiketsu-hen'	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,final_score,age_penalty
21051,,Mahou Shoujo Madoka★Magica Movie 4: Walpurgis no Kaiten,48820,Movie,"[drama, suspense, mystery, supernatural]",6.39,6.387130,1.088235,0.000000
16445,,Princess Principal: Crown Handler Movie 3,41141,Movie,"[action, mystery]",6.39,6.387130,1.055164,1.991667
21930,,Mahoutsukai no Yoru,50668,Movie,"[mystery, supernatural]",6.39,6.387130,1.055164,1.991667
22375,,City Hunter Movie: Tenshi no Namida,51585,Movie,"[action, mystery]",6.39,6.387130,1.055164,1.991667
14410,,Princess Principal: Crown Handler Movie 1,37807,Movie,"[action, mystery]",7.52,7.311564,0.982313,3.983333
16444,,Princess Principal: Crown Handler Movie 2,41140,Movie,"[action, mystery]",7.71,7.316857,0.982012,3.983333
13907,,Zoku Owarimonogatari,36999,Movie,"[comedy, mystery, supernatural]",8.45,8.414520,0.958951,6.825000
11051,,Kizumonogatari II: Nekketsu-hen,31757,Movie,"[action, mystery, supernatural]",8.58,8.565061,0.956823,8.962500
5670,,Kizumonogatari I: Tekketsu-hen,9260,Movie,"[action, mystery, supernatural]",8.37,8.357224,0.897322,8.962500
12647,,Bungou Stray Dogs: Dead Apple,34944,Movie,"[action, mystery, supernatural]",7.92,7.901468,0.870743,6.970833


### **Now lets add the Collaborative filtering**:

In [ ]:
class AnimeRecommender1:
    def __init__(self, anime_df, ratings_df):
        self.nlp = spacy.load("en_core_web_sm")
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self.nlp = spacy.load("en_core_web_sm")
        self.current_year = datetime.now().year

        self._preprocess_data()
        self._create_feature_matrices()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)
        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')
        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)
        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        def normalize_genre_list(genre_string):
            if genre_string == 'UNKNOWN' or pd.isna(genre_string):
                return []
            genres = re.split(r',\s*', genre_string)
            cleaned = [re.sub(r'[^\w\s]', '', g).strip().lower() for g in genres]
            return list(set(cleaned))

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(normalize_genre_list)
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(normalize_genre_list)

        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series: pd.Series) -> pd.Series:
        return text_series.apply(self._spacy_clean)

    def _spacy_clean(self, text: str) -> str:
        doc = self.nlp(text.lower())
        return ' '.join(
            token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct and token.is_alpha
        )

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        year_range = max_year - min_year if max_year != min_year else 1
        scaled_year = (year - min_year) / year_range
        return scaled_year

    def _calculate_anime_similarity(self, idx: int):
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        num_recs_to_get = min(52, len(self.anime_df) - 1)
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:num_recs_to_get + 1]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_anime_recommendations1(self, title, user_id, n=10,best_svd_model=SVD_model, similarity_weight=0.85, recency_weight=0.2, svd_weight=0.6):
      matching_animes = self.anime_df[self.anime_df['Name'] == title]
      if matching_animes.empty:
        raise ValueError(f"Anime '{title}' not found in the database.")
      idx = matching_animes.index[0]

      sim_scores_top, top_indices = self._calculate_anime_similarity(idx)

      recommended_animes = self.anime_df.iloc[top_indices][
        ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'Release_year', 'Episodes', 'Status']
      ].copy()

      recommended_animes['similarity_score'] = sim_scores_top
      recommended_animes['recency_score'] = recommended_animes['Release_year'].apply(self._calculate_recency_boost)
      recommended_animes['end_year'] = recommended_animes['Release_year'].fillna(self.current_year)
      recommended_animes['episodes'] = pd.to_numeric(recommended_animes['Episodes'], errors='coerce').fillna(self.median_episodes)

      recommended_animes['age_penalty'] = recommended_animes.apply(
        lambda row: 0 if row['Status'] == 'Currently Airing' else
        (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
        axis=1
        )

    # 🔄 Predict SVD ratings for this user if model and anime_id mapping exists
      svd_preds = []
      for anime_id in recommended_animes['anime_id']:
        try:
          pred = best_svd_model.predict(user_id, anime_id).est
        except Exception:
          pred = best_svd_model.trainset.global_mean  # fallback
        svd_preds.append(pred)

      recommended_animes['svd_pred'] = svd_preds

    # 🔄 Normalize weights
      rating_weight = max(0, 1 - similarity_weight)
      similarity_weight = max(0, similarity_weight)
      recency_weight = max(0, recency_weight)
      svd_weight = max(0, min(svd_weight, 1))
    # 🔄 Blend collaborative + content score
      content_score = (
        rating_weight * recommended_animes['weighted_rating'] +
        similarity_weight * recommended_animes['similarity_score']
    )

      recommended_animes['final_score'] = (
        ((1 - svd_weight) * content_score + svd_weight * recommended_animes['svd_pred']) * (1 - recency_weight) +
        recency_weight * recommended_animes['recency_score']
    ) / (1 + recommended_animes['age_penalty'] / 10)

      return recommended_animes.sort_values('final_score', ascending=False).head(n)[
        ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'svd_pred', 'final_score', 'age_penalty']
    ]

    def get_recommendations_by_genres_simple(self, genres, n=10):
        if isinstance(genres, str):
            genres = [genres]
        input_genres = {g.strip().lower() for g in genres}
        if not input_genres:
            raise ValueError("No valid genres provided.")

        # Always safely convert genre lists to sets on-the-fly
        genre_sets = self.anime_df['Genres'].apply(lambda g_list: {g.strip().lower() for g in g_list})
        self.anime_df['has_all_genres'] = genre_sets.apply(
            lambda anime_genres: input_genres.issubset(anime_genres)
        )

        recommendations = self.anime_df[self.anime_df['has_all_genres']].copy()
        current_year = self.current_year
        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        recommendations = recommendations.sort_values(
            by=['weighted_rating', 'age_penalty'], ascending=[False, True]
        ).head(n)

        return recommendations[['Image URL', 'Name', 'Genres', 'weighted_rating', 'age_penalty']]


* lets load the `SVD` model

In [ ]:
import pickle
with open('/content/drive/MyDrive/Anime Recommender System/svd_best_model13.pkl', 'rb') as f:
    SVD_model = pickle.load(f)


In [ ]:
recommender = AnimeRecommender1(anime_df, ratings_df)

In [ ]:
df = recommender.get_anime_recommendations1(
    title='One Piece',
    user_id=1,
    best_svd_model=SVD_model,
    n=24,
    similarity_weight=0.60,
    recency_weight=0.2,
    svd_weight=0.9
)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,svd_pred,final_score,age_penalty
23543,,Hirogaru Sky! Precure,53716,TV,"[fantasy, action]",7.69,6.923455,7.775836,6.043347,0.000000
21578,,Edens Zero 2nd Season,50002,TV,"[fantasy, adventure, scifi, action]",7.43,7.164798,7.302414,5.708149,0.000000
21472,,Fairy Tail: 100 Years Quest,49785,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,5.645024,0.000000
24174,,Boruto: Naruto Next Generations Part 2,54687,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,5.644798,0.000000
24042,,Ishura,54449,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,5.643434,0.000000
24186,,Mahoutsukai Precure! 2,54717,TV,"[fantasy, action]",6.39,6.387130,7.519458,5.315576,0.991667
23754,,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,53998,TV,"[fantasy, adventure, action]",6.39,6.387130,8.180449,5.271495,1.983333
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[fantasy, adventure, action]",7.74,7.643075,8.136377,4.895325,2.916667
24175,,Naruto (Shinsaku Anime),54688,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.881137,1.966667
22481,,Nanatsu no Taizai: Mokushiroku no Yonkishi,51794,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.874748,1.983333


In [ ]:
class AnimeRecommender1:
    def __init__(self, anime_df, ratings_df, svd_model):
        self.nlp = spacy.load("en_core_web_sm")
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self.svd_model = svd_model
        self.current_year = datetime.now().year

        self._preprocess_data()
        self._create_feature_matrices()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)
        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')
        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)
        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        def normalize_genre_list(genre_string):
            if genre_string == 'UNKNOWN' or pd.isna(genre_string):
                return []
            genres = re.split(r',\s*', genre_string)
            return list(set([re.sub(r'[^\w\s]', '', g).strip().lower() for g in genres]))

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(normalize_genre_list)
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(normalize_genre_list)

        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series):
        return text_series.apply(self._spacy_clean)

    def _spacy_clean(self, text):
        doc = self.nlp(text.lower())
        return ' '.join(token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha)

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        return (year - min_year) / (max_year - min_year) if max_year != min_year else 0

    def _calculate_anime_similarity(self, idx):
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:min(52, len(self.anime_df))]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_anime_recommendations1(self, title, user_id, n=10, similarity_weight=0.85, recency_weight=0.2, svd_weight=0.6):
        match = self.anime_df[self.anime_df['Name'] == title]
        if match.empty:
            raise ValueError(f"Anime '{title}' not found.")
        idx = match.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)
        recs = self.anime_df.iloc[top_indices].copy()
        recs['similarity_score'] = sim_scores_top
        recs['recency_score'] = recs['Release_year'].apply(self._calculate_recency_boost)
        recs['end_year'] = recs['Release_year'].fillna(self.current_year)
        recs['episodes'] = pd.to_numeric(recs['Episodes'], errors='coerce').fillna(self.median_episodes)

        recs['age_penalty'] = recs.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)), axis=1
        )

        svd_preds = []
        for anime_id in recs['anime_id']:
            try:
                pred = self.svd_model.predict(user_id, anime_id).est
            except Exception:
                pred = self.svd_model.trainset.global_mean
            svd_preds.append(pred)

        recs['svd_pred'] = svd_preds
        rating_weight = max(0, 1 - similarity_weight)
        content_score = rating_weight * recs['weighted_rating'] + similarity_weight * recs['similarity_score']

        recs['final_score'] = (
            ((1 - svd_weight) * content_score + svd_weight * recs['svd_pred']) * (1 - recency_weight) +
            recency_weight * recs['recency_score']
        ) / (1 + recs['age_penalty'] / 10)

        return recs.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'Genres', 'Score', 'weighted_rating', 'svd_pred', 'final_score', 'age_penalty']
        ]


In [ ]:
recommender = AnimeRecommender1(anime_df, ratings_df,svd_model=SVD_model)

In [ ]:
df = recommender.get_anime_recommendations1(
    title='Fairy Tail: 100 Years Quest',
    user_id=1,
    n=24
)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,svd_pred,final_score,age_penalty
11,,One Piece,21,TV,"[fantasy, adventure, action]",8.69,8.686696,8.614288,4.854554,0.000000
21578,,Edens Zero 2nd Season,50002,TV,"[fantasy, adventure, scifi, action]",7.43,7.164798,7.302414,4.188849,0.000000
24174,,Boruto: Naruto Next Generations Part 2,54687,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.065522,0.000000
24886,,Dekisokonai to Yobareta Motoeiyuu wa Jikka kara Tsuihou sareta node Sukikatte ni Ikiru Koto ni Shita,55717,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.058088,0.000000
22406,,Nozomanu Fushi no Boukensha,51648,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.056716,0.000000
23556,,"Sokushi Cheat ga Saikyou sugite, Isekai no Yatsura ga Marude Aite ni Naranai n desu ga.",53730,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,4.055115,0.000000
22758,,Ore dake Level Up na Ken,52299,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,3.873439,0.991667
23754,,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,53998,TV,"[fantasy, adventure, action]",6.39,6.387130,8.180449,3.821219,1.983333
24175,,Naruto (Shinsaku Anime),54688,TV,"[fantasy, adventure, action]",6.39,6.387130,7.519458,3.563505,1.966667
16329,,Dragon Quest: Dai no Daibouken (2020),40906,TV,"[fantasy, adventure, action]",7.74,7.643075,8.136377,3.563224,2.916667


In [ ]:
df = recommender.get_anime_recommendations1(
    title='Fairy Tail: 100 Years Quest',
    user_id=2,
    n=24
)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,Genres,Score,weighted_rating,svd_pred,final_score,age_penalty
11,,One Piece,21,TV,"[fantasy, adventure, action]",8.69,8.686696,9.239926,5.154860,0.000000
21578,,Edens Zero 2nd Season,50002,TV,"[fantasy, adventure, scifi, action]",7.43,7.164798,7.746519,4.402019,0.000000
24174,,Boruto: Naruto Next Generations Part 2,54687,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,4.272992,0.000000
24886,,Dekisokonai to Yobareta Motoeiyuu wa Jikka kara Tsuihou sareta node Sukikatte ni Ikiru Koto ni Shita,55717,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,4.265558,0.000000
22406,,Nozomanu Fushi no Boukensha,51648,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,4.264185,0.000000
23556,,"Sokushi Cheat ga Saikyou sugite, Isekai no Yatsura ga Marude Aite ni Naranai n desu ga.",53730,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,4.262585,0.000000
22758,,Ore dake Level Up na Ken,52299,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,4.062191,0.991667
23754,,Bleach: Sennen Kessen-hen - Ketsubetsu-tan,53998,TV,"[fantasy, adventure, action]",6.39,6.387130,8.528788,3.960748,1.983333
24175,,Naruto (Shinsaku Anime),54688,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,3.736878,1.966667
22481,,Nanatsu no Taizai: Mokushiroku no Yonkishi,51794,TV,"[fantasy, adventure, action]",6.39,6.387130,7.951687,3.731894,1.983333


In [ ]:
class AnimeRecommender1:
    def __init__(self, anime_df, ratings_df, svd_model):
        self.nlp = spacy.load("en_core_web_sm")
        self.anime_df = anime_df.copy()
        self.ratings_df = ratings_df.copy()
        self.svd_model = svd_model
        self.current_year = datetime.now().year

        self._preprocess_data()
        self._create_feature_matrices()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _preprocess_data(self):
        self.anime_df['Score'] = self.anime_df['Score'].replace('[^0-9.]', '', regex=True)
        self.anime_df['Scored By'] = self.anime_df['Scored By'].replace('[^0-9]', '', regex=True)
        self.anime_df['Score'] = pd.to_numeric(self.anime_df['Score'], errors='coerce')
        self.anime_df['Scored By'] = pd.to_numeric(self.anime_df['Scored By'], errors='coerce')
        self.anime_df['Score'].fillna(self.anime_df['Score'].median(), inplace=True)
        self.anime_df['Scored By'].fillna(self.anime_df['Scored By'].median(), inplace=True)
        self.anime_df['Release_year'] = self.anime_df['Aired'].str.extract(r'(\d{4})').astype(float)

        def normalize_genre_list(genre_string):
            if genre_string == 'UNKNOWN' or pd.isna(genre_string):
                return []
            genres = re.split(r',\s*', genre_string)
            return list(set([re.sub(r'[^\w\s]', '', g).strip().lower() for g in genres]))

        self.anime_df['Genres'] = self.anime_df['Genres'].apply(normalize_genre_list)
        self.anime_df['Studios'] = self.anime_df['Studios'].apply(normalize_genre_list)

        C = self.anime_df['Score'].mean()
        m = self.anime_df['Scored By'].quantile(0.65)
        self.anime_df['weighted_rating'] = (
            (self.anime_df['Scored By'] / (self.anime_df['Scored By'] + m)) * self.anime_df['Score'] +
            (m / (self.anime_df['Scored By'] + m)) * C
        )

    def _create_feature_matrices(self):
        self.mlb_genres = MultiLabelBinarizer(sparse_output=True)
        self.genres_encoded = self.mlb_genres.fit_transform(self.anime_df['Genres'])

        self.mlb_studios = MultiLabelBinarizer(sparse_output=True)
        self.studios_encoded = self.mlb_studios.fit_transform(self.anime_df['Studios'])

        self.ohe_type = OneHotEncoder(sparse_output=True)
        self.type_encoded = self.ohe_type.fit_transform(self.anime_df[['Type']])

        self.ohe_source = OneHotEncoder(sparse_output=True)
        self.source_encoded = self.ohe_source.fit_transform(self.anime_df[['Source']])

        self.anime_df['Episodes'] = pd.to_numeric(self.anime_df['Episodes'], errors='coerce')
        self.median_episodes = self.anime_df['Episodes'].median()
        self.anime_df['Episodes'].fillna(self.median_episodes, inplace=True)

        self.bins = [0, 1, 12, 24, 50, np.inf]
        self.labels = ['1', '2-12', '13-24', '25-50', '51+']
        self.anime_df['Episodes_Binned'] = pd.cut(self.anime_df['Episodes'], bins=self.bins, labels=self.labels)
        self.ohe_episodes = OneHotEncoder(sparse_output=True)
        self.episodes_encoded = self.ohe_episodes.fit_transform(self.anime_df[['Episodes_Binned']])

        self.anime_df['Synopsis'] = self._clean_text(self.anime_df['Synopsis'].fillna(''))
        self.tfidf = TfidfVectorizer(stop_words='english')
        self.synopsis_encoded = self.tfidf.fit_transform(self.anime_df['Synopsis'])

    def _clean_text(self, text_series):
        return text_series.apply(self._spacy_clean)

    def _spacy_clean(self, text):
        doc = self.nlp(text.lower())
        return ' '.join(token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha)

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        return (year - min_year) / (max_year - min_year) if max_year != min_year else 0

    def _calculate_anime_similarity(self, idx):
        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:min(52, len(self.anime_df))]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_anime_recommendations1(self, title, user_id, n=10, similarity_weight=0.85, recency_weight=0.2):
        match = self.anime_df[self.anime_df['Name'] == title]
        if match.empty:
            raise ValueError(f"Anime '{title}' not found.")
        idx = match.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)
        recs = self.anime_df.iloc[top_indices].copy()
        recs['similarity_score'] = sim_scores_top
        recs['recency_score'] = recs['Release_year'].apply(self._calculate_recency_boost)
        recs['end_year'] = recs['Release_year'].fillna(self.current_year)
        recs['episodes'] = pd.to_numeric(recs['Episodes'], errors='coerce').fillna(self.median_episodes)

        recs['age_penalty'] = recs.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)), axis=1
        )

        if user_id not in set(self.ratings_df['user_id']):
            # Cold-start fallback: no SVD available, use content-based only
            content_score = (
                (1 - similarity_weight) * recs['weighted_rating'] + similarity_weight * recs['similarity_score']
            )
            recs['final_score'] = (
                content_score * (1 - recency_weight) + recs['recency_score'] * recency_weight
            ) / (1 + recs['age_penalty'] / 10)
            recs['svd_pred'] = np.nan
        else:
            svd_preds = []
            for anime_id in recs['anime_id']:
                try:
                    pred = self.svd_model.predict(user_id, anime_id).est
                except Exception:
                    pred = self.svd_model.trainset.global_mean
                svd_preds.append(pred)

            user_rating_count = self.ratings_df[self.ratings_df['user_id'] == user_id].shape[0]
            if user_rating_count < 10:
              svd_weight = 0.5
            elif user_rating_count < 50:
              svd_weight = 0.6
            else:
              svd_weight = 0.7

            recs['svd_pred'] = svd_preds
            rating_weight = max(0, 1 - similarity_weight)
            content_score = rating_weight * recs['weighted_rating'] + similarity_weight * recs['similarity_score']

            recs['final_score'] = (
                ((1 - svd_weight) * content_score + svd_weight * recs['svd_pred']) * (1 - recency_weight) +
                recency_weight * recs['recency_score']
            ) / (1 + recs['age_penalty'] / 10)

        return recs.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'svd_pred', 'final_score']
        ]
    def get_recommendations_by_genres_simple(self, genres, n=10):
      if isinstance(genres, str):
        genres = [genres]

      input_genres = {g.strip().lower() for g in genres}
      if not input_genres:
        raise ValueError("No valid genres provided.")

      # Normalize genres per anime
      genre_sets = self.anime_df['Genres'].apply(lambda g_list: {g.strip().lower() for g in g_list})
      self.anime_df['has_all_genres'] = genre_sets.apply(
        lambda anime_genres: input_genres.issubset(anime_genres)
      )

      recommendations = self.anime_df[self.anime_df['has_all_genres']].copy()
      current_year = self.current_year

      recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
      recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

      # 🔥 Stronger age penalty
      recommendations['age_penalty'] = recommendations.apply(
        lambda row: 0 if row['Status'] == 'Currently Airing' else
        (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
        axis=1
    )

      recommendations = recommendations.sort_values(
        by=['weighted_rating', 'age_penalty'], ascending=[False, True]
        ).head(n)
      return recommendations[['Image URL', 'Name', 'Genres', 'weighted_rating', 'age_penalty']]



In [ ]:
anime_df = pd.read_csv('/content/drive/MyDrive/Anime Recommender System/anime_filtered.csv')
recommender = AnimeRecommender1(anime_df, ratings_df, svd_model=SVD_model)

In [ ]:
df = recommender.get_anime_recommendations1(
    title='Kimetsu no Yaiba: Yuukaku-hen',
    user_id=1,
    n=24
)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,svd_pred,final_score
23543,,Hirogaru Sky! Precure,53716,TV,7.775836,4.904126
21905,,Yu☆Gi☆Oh! Go Rush!!,50607,TV,7.334715,4.626674
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,7.519458,4.573433
23669,,Ao no Exorcist (Shin Series),53889,TV,7.519458,4.553033
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,8.193908,4.399936
21793,,Mato Seihei no Slave,50392,TV,7.519458,4.319778
20893,,"Maou Gakuin no Futekigousha: Shijou Saikyou no Maou no Shiso, Tensei shite Shison-tachi no Gakkou e Kayou II",48417,TV,6.535789,4.212508
22042,,Jujutsu Kaisen 2nd Season,51009,TV,7.853121,4.119205
18172,,Chainsaw Man,44511,TV,8.046383,3.986269
22190,,Ragna Crimson,51297,TV,7.519458,3.963279


In [ ]:
df = recommender.get_anime_recommendations1(
    title='Kimetsu no Yaiba: Yuukaku-hen',
    user_id=4,
    n=24
)
df
# df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

# display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,svd_pred,final_score
23543,https://cdn.myanimelist.net/images/anime/1762/...,Hirogaru Sky! Precure,53716,TV,7.597894,4.805134
21905,https://cdn.myanimelist.net/images/anime/1624/...,Yu☆Gi☆Oh! Go Rush!!,50607,TV,7.136368,4.515923
24875,https://cdn.myanimelist.net/images/anime/1230/...,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,7.079678,4.327156
23669,https://cdn.myanimelist.net/images/anime/1187/...,Ao no Exorcist (Shin Series),53889,TV,7.079678,4.306756
21685,https://cdn.myanimelist.net/images/anime/1926/...,Seiken Gakuin no Makentsukai,50184,TV,7.079678,4.297561
21793,https://cdn.myanimelist.net/images/anime/1406/...,Mato Seihei no Slave,50392,TV,7.079678,4.097866
22048,https://cdn.myanimelist.net/images/anime/1765/...,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,7.531537,4.092264
22042,https://cdn.myanimelist.net/images/anime/1600/...,Jujutsu Kaisen 2nd Season,51009,TV,7.485360,3.947345
18172,https://cdn.myanimelist.net/images/anime/1806/...,Chainsaw Man,44511,TV,7.947433,3.943268
20893,https://cdn.myanimelist.net/images/anime/1475/...,Maou Gakuin no Futekigousha: Shijou Saikyou no...,48417,TV,5.865756,3.838076


In [ ]:
df = recommender.get_anime_recommendations1(
    title='Kimetsu no Yaiba: Yuukaku-hen',
    user_id=499999999,
    n=24
)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,svd_pred,final_score
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,NaN,1.437633
20893,,"Maou Gakuin no Futekigousha: Shijou Saikyou no Maou no Shiso, Tensei shite Shison-tachi no Gakkou e Kayou II",48417,TV,NaN,1.383529
23543,,Hirogaru Sky! Precure,53716,TV,NaN,1.374168
21905,,Yu☆Gi☆Oh! Go Rush!!,50607,TV,NaN,1.277073
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,NaN,1.245945
18172,,Chainsaw Man,44511,TV,NaN,1.245837
21793,,Mato Seihei no Slave,50392,TV,NaN,1.208660
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,NaN,1.208456
21207,,High Card,49154,TV,NaN,1.169196
23669,,Ao no Exorcist (Shin Series),53889,TV,NaN,1.140456


In [ ]:
df = recommender.get_recommendations_by_genres_simple([ 'scifi','action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty
9880,,Gintama°,"[scifi, comedy, action]",9.040355,12.481034
5989,,Gintama',"[scifi, comedy, action]",9.019494,18.689781
7240,,Gintama': Enchousen,"[scifi, comedy, action]",9.000788,20.537456
15525,,Gintama: The Final,"[drama, comedy, action, scifi]",8.968517,5.256040
12179,,Gintama.,"[scifi, comedy, action]",8.947376,11.519446
833,,Gintama,"[scifi, comedy, action]",8.928261,17.118840
2647,,Code Geass: Hangyaku no Lelouch R2,"[drama, award winning, scifi, action]",8.906124,26.838975
7228,,Gintama Movie 2: Kanketsu-hen - Yorozuya yo Eien Nare,"[scifi, comedy, action]",8.876192,19.642834
14206,,Gintama.: Shirogane no Tamashii-hen - Kouhan-sen,"[scifi, comedy, action]",8.831842,9.727805
13818,,Gintama.: Shirogane no Tamashii-hen,"[scifi, comedy, action]",8.767080,9.813892


In [ ]:
import os
import joblib
import spacy
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import load_npz, save_npz
from datetime import datetime

class AnimeRecommender1:
    def __init__(self, ratings_df, svd_model, raw_anime_csv="/content/drive/MyDrive/Anime Recommender System/anime-dataset-2023.csv",
                 cache_dir="/content/drive/MyDrive/Anime Recommender System/cache"):
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
        self.ratings_df = ratings_df.copy()
        self.svd_model = svd_model
        self.current_year = datetime.now().year
        self.cache_dir = cache_dir
        self.raw_anime_csv = raw_anime_csv

        os.makedirs(cache_dir, exist_ok=True)
        self._ensure_cache()

        self.weights = {
            'genres': 0.30,
            'synopsis': 0.35,
            'type': 0.15,
            'studios': 0.10,
            'episodes': 0.05,
            'source': 0.05
        }

    def _ensure_cache(self):
        cd = self.cache_dir
        cache_files = [
            "anime_df_cleaned.pkl", "tfidf_matrix.npz", "tfidf_vocab.pkl",
            "genres_encoded.npz", "studios_encoded.npz", "type_encoded.npz",
            "source_encoded.npz", "episodes_encoded.npz"
        ]
        all_exist = all(os.path.exists(os.path.join(cd, f)) for f in cache_files)
        if not all_exist:
            print("🛠️ Generating and caching features (first run)...")
            self._generate_and_cache_all()
        else:
            print("🔁 Loading cached anime data and feature matrices...")
        self._load_cached_data()

    def _generate_and_cache_all(self):
        cd = self.cache_dir
        anime_df = pd.read_csv(self.raw_anime_csv)

        def fast_normalize(col):
            return col.fillna('').str.lower().str.replace(r'[^\w\s,]', '', regex=True).str.split(',\s*')

        anime_df['Score'] = pd.to_numeric(anime_df['Score'].str.replace('[^0-9.]', '', regex=True), errors='coerce')
        anime_df['Scored By'] = pd.to_numeric(anime_df['Scored By'].str.replace('[^0-9]', '', regex=True), errors='coerce')
        anime_df['Score'].fillna(anime_df['Score'].median(), inplace=True)
        anime_df['Scored By'].fillna(anime_df['Scored By'].median(), inplace=True)
        anime_df['Release_year'] = pd.to_numeric(anime_df['Aired'].str.extract(r'(\d{4})')[0], errors='coerce')

        anime_df['Genres'] = fast_normalize(anime_df['Genres']).apply(lambda lst: list(set([g.strip() for g in lst if g != ''])))
        anime_df['Studios'] = fast_normalize(anime_df['Studios']).apply(lambda lst: list(set([g.strip() for g in lst if g != ''])))

        C = anime_df['Score'].mean()
        m = anime_df['Scored By'].quantile(0.65)
        anime_df['weighted_rating'] = ((anime_df['Scored By'] / (anime_df['Scored By'] + m)) * anime_df['Score'] +
                                       (m / (anime_df['Scored By'] + m)) * C)

        mlb_genres = MultiLabelBinarizer(sparse_output=True)
        genres_encoded = mlb_genres.fit_transform(anime_df['Genres'])
        save_npz(os.path.join(cd, "genres_encoded.npz"), genres_encoded)

        mlb_studios = MultiLabelBinarizer(sparse_output=True)
        studios_encoded = mlb_studios.fit_transform(anime_df['Studios'])
        save_npz(os.path.join(cd, "studios_encoded.npz"), studios_encoded)

        ohe_type = OneHotEncoder(sparse_output=True)
        type_encoded = ohe_type.fit_transform(anime_df[['Type']])
        save_npz(os.path.join(cd, "type_encoded.npz"), type_encoded)

        ohe_source = OneHotEncoder(sparse_output=True)
        source_encoded = ohe_source.fit_transform(anime_df[['Source']])
        save_npz(os.path.join(cd, "source_encoded.npz"), source_encoded)

        anime_df['Episodes'] = pd.to_numeric(anime_df['Episodes'], errors='coerce')
        median_episodes = anime_df['Episodes'].median()
        anime_df['Episodes'].fillna(median_episodes, inplace=True)

        bins = [0, 1, 12, 24, 50, np.inf]
        labels = ['1', '2-12', '13-24', '25-50', '51+']
        anime_df['Episodes_Binned'] = pd.cut(anime_df['Episodes'], bins=bins, labels=labels)

        ohe_episodes = OneHotEncoder(sparse_output=True)
        episodes_encoded = ohe_episodes.fit_transform(anime_df[['Episodes_Binned']])
        save_npz(os.path.join(cd, "episodes_encoded.npz"), episodes_encoded)

        anime_df['Synopsis'] = anime_df['Synopsis'].fillna('')
        anime_df['cleaned_synopsis'] = self._parallel_clean(anime_df['Synopsis'])
        joblib.dump(anime_df['cleaned_synopsis'], os.path.join(cd, "synopsis_cleaned.pkl"))

        tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
        tfidf_matrix = tfidf.fit_transform(anime_df['cleaned_synopsis'])
        save_npz(os.path.join(cd, "tfidf_matrix.npz"), tfidf_matrix)
        joblib.dump(tfidf.vocabulary_, os.path.join(cd, "tfidf_vocab.pkl"))
        joblib.dump(anime_df, os.path.join(cd, "anime_df_cleaned.pkl"))
        print("✅ All preprocessing completed and cached.")

    def _parallel_clean(self, texts):
        docs = list(self.nlp.pipe(texts, batch_size=64))
        return [
            ' '.join(token.lemma_ for token in doc if token.is_alpha and not token.is_stop and not token.is_punct)
            for doc in docs
        ]

    def _load_cached_data(self):
        cd = self.cache_dir
        self.anime_df = joblib.load(os.path.join(cd, "anime_df_cleaned.pkl"))
        self.synopsis_encoded = load_npz(os.path.join(cd, "tfidf_matrix.npz"))
        self.tfidf_vocab = joblib.load(os.path.join(cd, "tfidf_vocab.pkl"))
        self.genres_encoded = load_npz(os.path.join(cd, "genres_encoded.npz"))
        self.studios_encoded = load_npz(os.path.join(cd, "studios_encoded.npz"))
        self.type_encoded = load_npz(os.path.join(cd, "type_encoded.npz"))
        self.source_encoded = load_npz(os.path.join(cd, "source_encoded.npz"))
        self.episodes_encoded = load_npz(os.path.join(cd, "episodes_encoded.npz"))
        self.median_episodes = self.anime_df['Episodes'].median()
        print("✅ All cached data loaded successfully.")

    def _calculate_recency_boost(self, release_year):
        year = release_year if pd.notna(release_year) else self.anime_df['Release_year'].min()
        min_year = self.anime_df['Release_year'].min()
        max_year = self.anime_df['Release_year'].max()
        return (year - min_year) / (max_year - min_year) if max_year != min_year else 0

    def _calculate_anime_similarity(self, idx):
        from sklearn.metrics.pairwise import cosine_similarity

        genres_sim = cosine_similarity(self.genres_encoded[idx:idx+1], self.genres_encoded)[0]
        studios_sim = cosine_similarity(self.studios_encoded[idx:idx+1], self.studios_encoded)[0]
        synopsis_sim = cosine_similarity(self.synopsis_encoded[idx:idx+1], self.synopsis_encoded)[0]
        type_sim = (self.anime_df['Type'] == self.anime_df.iloc[idx]['Type']).astype(float).values
        source_sim = (self.anime_df['Source'] == self.anime_df.iloc[idx]['Source']).astype(float).values
        episodes_sim = (self.anime_df['Episodes_Binned'] == self.anime_df.iloc[idx]['Episodes_Binned']).astype(float).values

        combined_sim = (
            self.weights['genres'] * genres_sim +
            self.weights['synopsis'] * synopsis_sim +
            self.weights['type'] * type_sim +
            self.weights['studios'] * studios_sim +
            self.weights['episodes'] * episodes_sim +
            self.weights['source'] * source_sim
        )

        sim_scores = np.array(list(enumerate(combined_sim)))
        sorted_sim_scores = sim_scores[sim_scores[:, 1].argsort()[::-1]]
        top_sim = sorted_sim_scores[1:min(52, len(self.anime_df))]

        top_indices = top_sim[:, 0].astype(int)
        sim_scores_top = top_sim[:, 1]
        return sim_scores_top, top_indices

    def get_anime_recommendations1(self, title, user_id, n=10, similarity_weight=0.85, recency_weight=0.2):
        match = self.anime_df[self.anime_df['Name'] == title]
        if match.empty:
            raise ValueError(f"Anime '{title}' not found.")
        idx = match.index[0]

        sim_scores_top, top_indices = self._calculate_anime_similarity(idx)
        recs = self.anime_df.iloc[top_indices].copy()
        recs['similarity_score'] = sim_scores_top
        recs['recency_score'] = recs['Release_year'].apply(self._calculate_recency_boost)
        recs['end_year'] = recs['Release_year'].fillna(self.current_year)
        recs['episodes'] = pd.to_numeric(recs['Episodes'], errors='coerce').fillna(self.median_episodes)

        recs['age_penalty'] = recs.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (self.current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)), axis=1
        )

        if user_id not in set(self.ratings_df['user_id']):
            content_score = (
                (1 - similarity_weight) * recs['weighted_rating'] + similarity_weight * recs['similarity_score']
            )
            recs['final_score'] = (
                content_score * (1 - recency_weight) + recs['recency_score'] * recency_weight
            ) / (1 + recs['age_penalty'] / 10)
            recs['svd_pred'] = np.nan
        else:
            svd_preds = []
            for anime_id in recs['anime_id']:
                try:
                    pred = self.svd_model.predict(user_id, anime_id).est
                except Exception:
                    pred = self.svd_model.trainset.global_mean
                svd_preds.append(pred)

            user_rating_count = self.ratings_df[self.ratings_df['user_id'] == user_id].shape[0]
            if user_rating_count < 10:
                svd_weight = 0.5
            elif user_rating_count < 50:
                svd_weight = 0.6
            else:
                svd_weight = 0.7

            recs['svd_pred'] = svd_preds
            rating_weight = max(0, 1 - similarity_weight)
            content_score = rating_weight * recs['weighted_rating'] + similarity_weight * recs['similarity_score']

            recs['final_score'] = (
                ((1 - svd_weight) * content_score + svd_weight * recs['svd_pred']) * (1 - recency_weight) +
                recency_weight * recs['recency_score']
            ) / (1 + recs['age_penalty'] / 10)

        return recs.sort_values('final_score', ascending=False).head(n)[
            ['Image URL', 'Name', 'anime_id', 'Type', 'svd_pred', 'final_score']
        ]

    def get_recommendations_by_genres_simple(self, genres, n=10):
        if isinstance(genres, str):
            genres = [genres]

        input_genres = {g.strip().lower() for g in genres}
        if not input_genres:
            raise ValueError("No valid genres provided.")

        genre_sets = self.anime_df['Genres'].apply(lambda g_list: {g.strip().lower() for g in g_list})
        self.anime_df['has_all_genres'] = genre_sets.apply(
            lambda anime_genres: input_genres.issubset(anime_genres)
        )

        recommendations = self.anime_df[self.anime_df['has_all_genres']].copy()
        current_year = self.current_year

        recommendations['end_year'] = recommendations['Release_year'].fillna(current_year)
        recommendations['episodes'] = pd.to_numeric(recommendations['Episodes'], errors='coerce').fillna(self.median_episodes)

        recommendations['age_penalty'] = recommendations.apply(
            lambda row: 0 if row['Status'] == 'Currently Airing' else
            (current_year - row['end_year']) * (1 - 0.5 * min(row['episodes'] / 120, 1)),
            axis=1
        )

        recommendations['final_score'] = recommendations['weighted_rating'] / (1 + (recommendations['age_penalty'] / 10))

        recommendations = recommendations.sort_values(by='final_score', ascending=False).head(n)

        return recommendations[['Image URL', 'Name', 'Genres', 'weighted_rating', 'age_penalty', 'final_score']]


In [ ]:
recommender = AnimeRecommender1(ratings_df, svd_model=SVD_model)

🔁 Loading cached anime data and feature matrices...
✅ All cached data loaded successfully.


In [ ]:
df = recommender.get_anime_recommendations1(
    title='Fate/stay night: Unlimited Blade Works 2nd Season',
    user_id=20000000000,
    n=24
)
# df
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,svd_pred,final_score
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,51019,TV,NaN,1.304393
20430,,Kimetsu no Yaiba: Yuukaku-hen,47778,TV,NaN,1.149130
21533,,Kimetsu no Yaiba: Mugen Ressha-hen,49926,TV,NaN,1.110795
24875,,Kimetsu no Yaiba: Hashira Geiko-hen,55701,TV,NaN,1.103021
11630,,Fate/stay night Movie: Heaven's Feel - III. Spring Song,33050,Movie,NaN,1.068723
11629,,Fate/stay night Movie: Heaven's Feel - II. Lost Butterfly,33049,Movie,NaN,1.004636
16352,,Enen no Shouboutai: Ni no Shou,40956,TV,NaN,0.995372
14539,,Kimetsu no Yaiba,38000,TV,NaN,0.988310
14613,,Fate/Grand Order: Zettai Majuu Sensen Babylonia,38084,TV,NaN,0.976281
16089,,"Maou Gakuin no Futekigousha: Shijou Saikyou no Maou no Shiso, Tensei shite Shison-tachi no Gakkou e Kayou",40496,TV,NaN,0.946744


In [ ]:
df = recommender.get_recommendations_by_genres_simple([ 'scifi','romance', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty,final_score
21951,,Mahouka Koukou no Rettousei (Zoku-hen),"[fantasy, action, scifi, romance]",6.387130,0.000000,6.387130
22702,,Date A Live V,"[action, scifi, romance]",6.387130,0.000000,6.387130
24772,,Macross (Shinsaku Animation),"[action, scifi, romance]",6.387130,0.000000,6.387130
16612,,Date A Live IV,"[action, scifi, romance]",7.746146,2.850000,6.028129
16090,,Mahouka Koukou no Rettousei: Raihousha-hen,"[fantasy, action, scifi, romance]",7.269632,4.729167,4.935535
20157,,Gekijou Tanpen Macross Frontier: Toki no Meikyuu,"[action, scifi, romance]",6.541423,3.983333,4.678014
20436,,Ai Zai Xiyuan Qian 2nd Season,"[fantasy, action, scifi, romance]",6.386648,3.733333,4.650471
13676,,Date A Live III,"[action, scifi, romance]",7.182833,5.700000,4.575053
13216,,Darling in the FranXX,"[action, drama, scifi, romance]",7.208504,6.300000,4.422395
17902,,Chu Feng: Yi Dian Zhi Zi,"[fantasy, action, scifi, romance]",6.346420,4.750000,4.302657


In [ ]:
df = recommender.get_recommendations_by_genres_simple(['fantasy', 'adventure', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty,final_score
11,,One Piece,"[fantasy, action, adventure]",8.686696,0.000000,8.686696
19600,,Jigokuraku,"[fantasy, action, adventure]",8.224549,0.000000,8.224549
21578,,Edens Zero 2nd Season,"[fantasy, action, scifi, adventure]",7.164798,0.000000,7.164798
23659,,Pokemon (2023),"[fantasy, action, adventure, comedy]",7.112476,0.000000,7.112476
16617,,Bleach: Sennen Kessen-hen,"[fantasy, action, adventure]",9.048079,2.837500,7.048163
22054,,Doupo Cangqiong: Nian Fan,"[fantasy, action, adventure]",6.968240,0.000000,6.968240
21385,,Tunshi Xingkong 2nd Season,"[fantasy, action, scifi, adventure]",6.952457,0.000000,6.952457
23239,,Dungeon ni Deai wo Motomeru no wa Machigatteiru Darou ka IV: Fuka Shou - Yakusai-hen,"[fantasy, action, adventure]",8.199359,1.908333,6.885396
20156,,Wanmei Shijie,"[fantasy, action, adventure]",6.836767,0.000000,6.836767
22968,,Shen Yin Wangzuo 2nd Season,"[fantasy, action, adventure]",6.798776,0.000000,6.798776


In [ ]:
df = recommender.get_recommendations_by_genres_simple(['horror', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty,final_score
14170,,Chimera,"[horror, action, supernatural]",6.387130,0.000000,6.387130
23140,,Berserk: Ougon Jidai-hen - Memorial Edition,"[fantasy, adventure, action, drama, horror]",7.673791,2.837500,5.977636
14942,,Dorohedoro,"[fantasy, action, comedy, horror]",8.048611,4.750000,5.456685
23844,,Zom 100: Zombie ni Naru made ni Shitai 100 no Koto,"[supernatural, suspense, action, comedy, horror]",6.387130,1.983333,5.330011
24008,,Biohazard: Death Island,"[horror, action, scifi]",6.387130,1.991667,5.326307
23881,,Muja,"[horror, action]",6.387130,2.987500,4.917906
23021,,Kinemaquia PV,"[horror, action, supernatural]",6.343686,2.987500,4.884455
17645,,Tenkuu Shinpan,"[horror, action, mystery]",6.706470,3.800000,4.859761
12419,,Koutetsujou no Kabaneri Movie 3: Unato Kessen,"[fantasy, action, drama, horror]",7.653065,5.975000,4.790651
16394,,Dorohedoro: Ma no Omake,"[fantasy, action, comedy, horror]",7.055402,4.875000,4.743128


In [ ]:
df = recommender.get_recommendations_by_genres_simple([ 'romance','action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty,final_score
16770,,Ling Jian Zun 4th Season,"[fantasy, action, adventure, romance]",6.578649,0.000000,6.578649
21268,,Wan Jie Xian Zong 5th Season,"[fantasy, action, romance]",6.487966,0.000000,6.487966
22469,,Xingchen Bian 5th Season,"[fantasy, action, adventure, romance]",6.476171,0.000000,6.476171
23948,,Yao Shen Ji 6th Season,"[fantasy, action, adventure, romance]",6.456534,0.000000,6.456534
22139,,Long Wang Dian 2nd Season,"[action, romance, comedy]",6.387130,0.000000,6.387130
24777,,Si Ge Yongzhe,"[fantasy, adventure, action, comedy, romance]",6.387130,0.000000,6.387130
24775,,Ruler of the Land,"[fantasy, action, drama, comedy, romance, ecchi]",6.387130,0.000000,6.387130
24828,,MY WIFE IS A DEMON QUEEN,"[fantasy, action, adventure, romance]",6.387130,0.000000,6.387130
22702,,Date A Live V,"[action, scifi, romance]",6.387130,0.000000,6.387130
22734,,Baozou Xia Ri,"[action, romance, comedy]",6.387130,0.000000,6.387130


In [ ]:
df = recommender.get_recommendations_by_genres_simple(['fantasy', 'action']	, n=50)
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,Genres,weighted_rating,age_penalty,final_score
11,,One Piece,"[fantasy, action, adventure]",8.686696,0.000000,8.686696
19600,,Jigokuraku,"[fantasy, action, adventure]",8.224549,0.000000,8.224549
22709,,Mashle,"[fantasy, action, comedy]",7.553219,0.000000,7.553219
22090,,NieR:Automata Ver1.1a,"[fantasy, action, scifi]",7.355421,0.000000,7.355421
13312,,Fate/Grand Order,"[fantasy, action]",7.285004,0.000000,7.285004
23475,,Dead Mount Death Play,"[fantasy, action, supernatural]",7.240476,0.000000,7.240476
21578,,Edens Zero 2nd Season,"[fantasy, action, scifi, adventure]",7.164798,0.000000,7.164798
23659,,Pokemon (2023),"[fantasy, action, adventure, comedy]",7.112476,0.000000,7.112476
22048,,Kimetsu no Yaiba: Katanakaji no Sato-hen,"[fantasy, action]",8.467293,1.908333,7.110393
16617,,Bleach: Sennen Kessen-hen,"[fantasy, action, adventure]",9.048079,2.837500,7.048163


In [ ]:
recommender = AnimeRecommender1(ratings_df, svd_model=SVD_model)

🛠️ Generating and caching features (first run)...
✅ All preprocessing completed and cached.
✅ All cached data loaded successfully.


In [ ]:
df = recommender.get_anime_recommendations1(
    title='Darling in the FranXX',
    user_id=1,
    n=24
)
# df
df['Image URL'] = df['Image URL'].apply(lambda x: f'<img src="{x}" width="100"/>')

display(HTML(df.to_html(escape=False)))

,Image URL,Name,anime_id,Type,svd_pred,final_score
21948,,Lycoris Recoil,50709,TV,7.479177,3.720411
20964,,86 Part 2,48569,TV,7.972586,3.672763
16608,,86,41457,TV,7.866722,3.614240
16309,,SSSS.Dynazenon,40870,TV,7.568760,3.475140
21073,,Cardfight!! Vanguard: overDress Season 2,48862,TV,7.434672,3.402961
22263,,Engage Kiss,51417,TV,6.319085,3.177520
16266,,Hypnosis Mic: Division Rap Battle - Rhyme Anima,40803,TV,7.165139,3.086045
13591,,Beatless,36516,TV,6.985848,2.695937
11508,,Senki Zesshou Symphogear AXZ,32836,TV,7.403490,2.673891
8855,,Shigatsu wa Kimi no Uso,23273,TV,8.366180,2.638317
